# Trend Tracker — Notebook 03: Insights Generation

Runs the full analytical pipeline on the enriched corpus, from TF-IDF and NMF
topic discovery through LLM synthesis, metadata-aware structured extraction,
verification, packaging, and DOCX report generation.

**Run order:** `01_token_preprocess` → `02_semantic_enrichment` → `03_insights_generation`

| Step | What | Key outputs |
|------|------|-------------|
| 1 | Build TF-IDF matrices | in-memory for Steps 2-4 |
| 2 | Quality checkpoint 2 | `quality/quality_cp2.json` |
| 3 | Category TF-IDF | `analysis/category_tfidf.csv` |
| 4 | NMF topic discovery | `analysis/nmf_topics.csv`, `analysis/project_topic_bridge.csv` |
| 5 | LLM topic labeling | `analysis/llm_topic_labels.json` |
| 6 | Synthesis | `analysis/llm_synthesis_*.txt` |
| 7 | Structured insight extraction with optional metadata scope signals | `insights/insights_candidates*.json`, `insights/metadata_lift_facts_for_step7.csv` |
| 8 | Topic verification | updates insights in-memory |
| 9 | Evidence tables | `insights/insights_flat.csv`, `insights/insight_topic_support_candidates.csv` |
| 10 | Packaging, dedupe & topline | updates curated insights in-memory |
| 11 | Final outputs & manifests | `reports/trend_tracker_report.docx`, `insights/insights_structured.json` |

Strategic-loop mode is optional. When enabled in `params.yaml`, this notebook
runs one resolved strategic area × split at a time. Existing NB03 filters still
define the corpus scope; strategic-loop preparation happens only after those
filters are applied.


---
## Setup and Configuration

In [ ]:
import sys
import json
import shutil
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed
from collections import defaultdict
from copy import deepcopy

import pandas as pd
import numpy as np
import math
import yaml
import re

ROOT = Path(".")
sys.path.insert(0, str(ROOT))

# Prefer a local utils.py when present. Otherwise load the most recently
# modified utils*.py file so versioned development utilities can be used
# without relying on filename sort order.
import importlib.util as _ilu, sys as _sys
if (ROOT / "utils.py").exists():
    _u = ROOT / "utils.py"
else:
    _utils_candidates = sorted(
        Path(".").glob("utils*.py"),
        key=lambda p: p.stat().st_mtime,
        reverse=True,
    )
    if not _utils_candidates:
        raise FileNotFoundError("No utils.py or utils*.py file found in project root.")
    _u = _utils_candidates[0]

if _u.stem != "utils":
    _s = _ilu.spec_from_file_location("utils", _u)
    _m = _ilu.module_from_spec(_s)
    _s.loader.exec_module(_m)
    _sys.modules["utils"] = _m

from utils import (
    load_cfg,
    write_json,
    artifact_meta,
    build_run_output_path,
    get_run_date,
    canonicalize_filter_spec,
    get_filter_fields_key,
    get_run_id,
    apply_filters,
    ensure_warning_file,
    append_warning,
    get_llm_client,
    start_stage_manifest,
    finalize_stage_manifest,
    build_pipeline_manifest,
    tokens_to_str,
    make_vec,
    upweight_injected_tokens,
    group_key,
    build_project_topic_bridge,
    quality_report,
    cat_tfidf_slice,
    nmf_one,
    build_input,
    _label_with_retry,
    build_cluster_input,
    _label_cluster_with_retry,
    coerce_token_list,
    _norm_group_value,
    _safe_topic_id,
    build_topic_lines,
    _call_with_retry,
    synthesize_one_group,
    strip_json_fences,
    normalize_insight,
    build_bridge_lookup,
    build_label_index,
    build_verified_insight_tables,
    apply_deterministic_packaging,
    dedupe_packaged_insights,
    assign_topline_sections_simple,
    build_structured_from_curated,
    build_looker_project_url,
    build_packaged_report_docx,
    project_insight_for_saved_candidates,
    _verify_insight_list,
    resolve_params_path,
    # Strategic-loop helpers
    prepare_strategic_loop_run_dataframe,
    add_strategic_derived_fields,
    expand_strategic_run_plan,
    candidate_support_project_ids_from_source_topics,
    build_metadata_lift_context,
    compute_metadata_lift,
    format_metadata_lift_facts,
    DEFAULT_METADATA_LIFT_DIMENSIONS,
    DEFAULT_METADATA_LIFT_THRESHOLDS,
)

CFG_PATH = resolve_params_path()
CFG = load_cfg(CFG_PATH)

try:
    MANIFEST_CFG_PATH = str(CFG_PATH.relative_to(ROOT))
except ValueError:
    MANIFEST_CFG_PATH = str(CFG_PATH)

ct = CFG["tfidf"]
client = get_llm_client()

# Load the enriched corpus produced by NB02.
raw_df = pd.read_parquet(ROOT / "OUTPUTS/prepared/06_enriched.parquet")

# Stopwords for the quality gate at Step 2. Loaded from params.yaml so the
# list can be extended without touching notebook code.
STOPWORDS = CFG["quality"]["stopword_violation_list"]

print(f"Loaded {len(raw_df):,} rows | {raw_df['project_id'].nunique():,} projects | {raw_df.shape[1]} columns")
print(f"Config path: {CFG_PATH}")
print(f"Utils path: {_u}")


---
## Parameters

All runtime configuration is resolved here from `params.yaml`. Edit
`params.yaml` to change behaviour; do not hardcode values in downstream cells.

Strategic-loop mode is intentionally small in `params.yaml`. The notebook expects
one resolved run at a time, for example:

```yaml
strategic_loop:
  enabled: true
  strategic_area_id: mental_health_sel
  split: strategic_injected_tag
  strategic_areas_path: CONFIG/strategic_areas.yaml
  state_clusters_path: CONFIG/state_clusters.yaml
```

No `run_scope` is used here. Existing NB03 filters define the analysis corpus.


In [ ]:
# ── Optional external strategic-loop config loading ──────────────────────────
# Strategic area definitions can live directly inside params.yaml, or in external
# YAML files referenced by strategic_loop.strategic_areas_path and
# strategic_loop.state_clusters_path. This keeps params small while still
# allowing fully resolved single-run configs.

def _read_yaml_if_present(path_value):
    if not path_value:
        return {}
    path = Path(path_value)
    if not path.is_absolute():
        path = ROOT / path
    if not path.exists():
        raise FileNotFoundError(f"Configured YAML path does not exist: {path}")
    with open(path, "r", encoding="utf-8") as f:
        return yaml.safe_load(f) or {}

CFG_RUNTIME = deepcopy(CFG)
STRATEGIC_LOOP_CFG = CFG_RUNTIME.get("strategic_loop", {}) or {}
STRATEGIC_LOOP_ENABLED = bool(STRATEGIC_LOOP_CFG.get("enabled", False))

if STRATEGIC_LOOP_ENABLED:
    strategic_areas_external = _read_yaml_if_present(
        STRATEGIC_LOOP_CFG.get("strategic_areas_path")
    )
    if "strategic_areas" in strategic_areas_external:
        CFG_RUNTIME["strategic_areas"] = strategic_areas_external["strategic_areas"]

    state_clusters_path = STRATEGIC_LOOP_CFG.get("state_clusters_path")
    state_clusters_external = _read_yaml_if_present(state_clusters_path)
    if state_clusters_path and "state_clusters" not in state_clusters_external:
        print(
            "WARNING: strategic_loop.state_clusters_path was provided, but the YAML "
            "does not contain a top-level 'state_clusters:' key. Falling back to "
            "utils.DEFAULT_STATE_CLUSTERS."
        )
    STATE_CLUSTERS = state_clusters_external.get("state_clusters")
else:
    STATE_CLUSTERS = None

print(f"Strategic loop enabled: {STRATEGIC_LOOP_ENABLED}")
if STRATEGIC_LOOP_ENABLED:
    print(f"Strategic area: {STRATEGIC_LOOP_CFG.get('strategic_area_id')}")
    print(f"Split: {STRATEGIC_LOOP_CFG.get('split')}")


In [ ]:
# ── Analysis scope ────────────────────────────────────────────────────────────
BASE_GROUPBY_FIELD = CFG["analysis"]["group_by"]
FILTER_LOGIC       = CFG["analysis"].get("filter_logic", "and")
FILTERS            = CFG["analysis"].get("filters", [])
EXCLUDE_GROUPS     = CFG["analysis"]["exclude_groups"]
REVIEW_GROUP       = CFG["analysis"]["review_group"]

# Apply declarative NB03 filters first. Strategic-loop prep happens after this,
# so no separate strategic run_scope/date filter exists.
df, filter_summary = apply_filters(raw_df, FILTER_LOGIC, FILTERS)
if filter_summary["no_rows_after_filter"]:
    raise ValueError("No rows remain after applying analysis filters. Check params.yaml.")

BASE_FILTERED_PROJECT_COUNT = int(df["project_id"].nunique())

# ── v1 feature guards ────────────────────────────────────────────────────────
# Binned/time-slice mode is still guarded out because topic identity and
# source_topic traceability are keyed by group + topic_id only.
if CFG["analysis"].get("bins", []):
    raise ValueError(
        "analysis.bins is non-empty: binned/time-slice mode is not supported "
        "in v1. Topic identity and source_topic traceability are not bin-aware. "
        "Set analysis.bins: [] to run without time bucketing."
    )

if REVIEW_GROUP is not None:
    raise ValueError(
        f"analysis.review_group is set to {REVIEW_GROUP!r}: single-group review "
        "mode is not supported in v1. Set analysis.review_group: null to run "
        "against all groups."
    )

# Legacy waterfall fields from older NB03 versions were derived in-notebook.
# v1.5 does not recreate them in legacy mode; use strategic_loop instead.
_REMOVED_LEGACY_DERIVED_GROUPBY_FIELDS = {
    "funding_status_x_metro",
    "efs_x_grade_band",
    "specialty_subject",
    "industry",
    "safety_justice",
    "future_framing",
    "framing_context",
    "current_pressures",
}
if not STRATEGIC_LOOP_ENABLED and BASE_GROUPBY_FIELD in _REMOVED_LEGACY_DERIVED_GROUPBY_FIELDS:
    raise ValueError(
        f"analysis.group_by={BASE_GROUPBY_FIELD!r} was a derived grouping used in older NB03 versions. "
        "Enable strategic_loop with the equivalent strategic area/split, or change analysis.group_by "
        "to a physical column in OUTPUTS/prepared/06_enriched.parquet."
    )

# Add the same standard derived fields in legacy mode that strategic-mode prep adds.
# This preserves v1.5.1 metadata-lift behavior and lets legacy runs group on
# physical-or-standard-derived columns such as project_cost_bucket or state_cluster.
if not STRATEGIC_LOOP_ENABLED:
    df = add_strategic_derived_fields(df, state_clusters=STATE_CLUSTERS)

if not STRATEGIC_LOOP_ENABLED and BASE_GROUPBY_FIELD not in df.columns:
    raise ValueError(
        f"analysis.group_by={BASE_GROUPBY_FIELD!r} is not present in the filtered enriched dataframe. "
        "Use a physical column or enable strategic_loop with a resolved split."
    )

def _validate_selected_area_tag_columns(cfg_runtime, strategic_area_id, frame, *, allow_missing=False):
    """Validate selected-area tag_* schema before strategic prep.

    Strict mode is the default. allow_missing=True lets exploratory runs proceed
    after dropping missing tags from this run's area spec, but still fails if no
    tags remain available.
    """
    strategic_areas = cfg_runtime.get("strategic_areas", {}) or {}
    if strategic_area_id not in strategic_areas:
        raise ValueError(f"Unknown strategic_area_id {strategic_area_id!r} in strategic_loop config.")

    area_spec = deepcopy(strategic_areas[strategic_area_id] or {})
    include_tags = area_spec.get("include_taxonomy_tags", []) or []
    if not include_tags:
        raise ValueError(
            f"Strategic area {strategic_area_id!r} has no include_taxonomy_tags. "
            "NB03 strategic runs currently require tag_* boolean membership columns."
        )

    available_tags = [tag for tag in include_tags if f"tag_{tag}" in frame.columns]
    missing_cols = [f"tag_{tag}" for tag in include_tags if f"tag_{tag}" not in frame.columns]

    if missing_cols and not allow_missing:
        raise ValueError(
            f"Strategic area {strategic_area_id!r} is missing {len(missing_cols)} required tag columns "
            f"in 06_enriched.parquet. Example missing columns: {missing_cols[:10]}. "
            "Re-run NB02 with the current taxonomy, update strategic_areas.yaml, or set "
            "strategic_loop.allow_missing_tag_columns: true for exploratory soft-fail mode."
        )

    if missing_cols and allow_missing:
        print(
            f"WARNING: Strategic area {strategic_area_id!r} is missing {len(missing_cols)} tag columns; "
            f"dropping them for this run because strategic_loop.allow_missing_tag_columns=true. "
            f"Example missing columns: {missing_cols[:10]}"
        )
        if not available_tags:
            raise ValueError(
                f"Strategic area {strategic_area_id!r} has no available tag_* columns after dropping missing tags."
            )
        area_spec["include_taxonomy_tags"] = available_tags

    return area_spec

# ── Strategic-loop dataframe preparation ─────────────────────────────────────
# The prep helper handles:
# - membership from tag_{taxonomy_tag} columns
# - derived fields: state_cluster, project_cost_bucket, funding_status, posting_period
# - project_category_bucketed with Missing/Awaiting Classification -> All Other
# - strategic_injected_tag explosion
# - 200-project hard group minimum, unless overridden in config
# - removal of area-defining injected tokens for strategic_injected_tag only
if STRATEGIC_LOOP_ENABLED:
    strategic_area_id = STRATEGIC_LOOP_CFG.get("strategic_area_id")
    split_spec = STRATEGIC_LOOP_CFG.get("split")
    if not strategic_area_id or split_spec is None:
        raise ValueError(
            "strategic_loop.enabled is true, but strategic_area_id or split is missing."
        )

    area_spec = _validate_selected_area_tag_columns(
        CFG_RUNTIME,
        strategic_area_id,
        raw_df,
        allow_missing=bool(STRATEGIC_LOOP_CFG.get("allow_missing_tag_columns", False)),
    )

    # Keep NB03 one-run-at-a-time and avoid scanning every strategic area's tag columns.
    CFG_FOR_THIS_RUN = deepcopy(CFG_RUNTIME)
    CFG_FOR_THIS_RUN["strategic_areas"] = {strategic_area_id: area_spec}

    project_category_bucket_rules = (
        STRATEGIC_LOOP_CFG.get("project_category_bucket_rules")
        or CFG_RUNTIME.get("project_category_bucket_rules")
        or {}
    )
    min_group_projects = int(
        STRATEGIC_LOOP_CFG.get(
            "min_projects_per_group",
            STRATEGIC_LOOP_CFG.get(
                "min_projects_per_strategic_injected_tag_group",
                CFG_RUNTIME.get("defaults", {}).get("min_projects_per_strategic_injected_tag_group", 200),
            ),
        )
    )

    df, STRATEGIC_RUN_META = prepare_strategic_loop_run_dataframe(
        df,
        cfg=CFG_FOR_THIS_RUN,
        strategic_area_id=strategic_area_id,
        split_spec=split_spec,
        min_group_projects=min_group_projects,
        project_category_bucket_rules=project_category_bucket_rules,
        state_clusters=STATE_CLUSTERS,
    )
    GROUPBY_FIELD = STRATEGIC_RUN_META["groupby_field"]
else:
    GROUPBY_FIELD = BASE_GROUPBY_FIELD
    STRATEGIC_RUN_META = {
        "strategic_loop_enabled": False,
        "strategic_area_id": None,
        "strategic_area_label": None,
        "split_id": None,
        "groupby_fields": [GROUPBY_FIELD],
        "groupby_field": GROUPBY_FIELD,
        "is_strategic_injected_tag": False,
        "min_group_projects": None,
        "input_project_count": BASE_FILTERED_PROJECT_COUNT,
        "area_project_count": None,
        "run_row_count": int(len(df)),
        "run_project_count": int(df["project_id"].nunique()),
        "group_count": int(df[GROUPBY_FIELD].nunique(dropna=True)),
        "group_counts": [],
        "removed_area_defining_injected_tokens": False,
    }

# ── Group scope: apply exclude_groups once, after the final groupby is resolved ─
if EXCLUDE_GROUPS:
    n_before = len(df)
    df = df[
        ~df[GROUPBY_FIELD].astype(str).isin([str(g) for g in EXCLUDE_GROUPS])
    ].reset_index(drop=True)
    if df.empty:
        raise ValueError(
            f"No rows remain after excluding groups {EXCLUDE_GROUPS}. "
            "Check analysis.exclude_groups in params.yaml."
        )
    print(
        f"exclude_groups: {n_before - len(df):,} rows removed "
        f"({len(EXCLUDE_GROUPS)} group(s): {EXCLUDE_GROUPS})"
    )
else:
    df = df.reset_index(drop=True)

# Refresh meta counts after exclude_groups so printed metadata and manifests match the actual run dataframe.
STRATEGIC_RUN_META["run_row_count"] = int(len(df))
STRATEGIC_RUN_META["run_project_count"] = int(df["project_id"].nunique())
STRATEGIC_RUN_META["group_count"] = int(df[GROUPBY_FIELD].nunique(dropna=True))

_fresh_group_counts = (
    df[[GROUPBY_FIELD, "project_id"]]
    .dropna(subset=[GROUPBY_FIELD, "project_id"])
    .drop_duplicates()
    .groupby(GROUPBY_FIELD)["project_id"]
    .nunique()
    .rename("project_count")
    .reset_index()
    .sort_values("project_count", ascending=False)
)

# Preserve group-level audit trails whenever strategic prep supplied one.
# This applies to both strategic_injected_tag and non-tag strategic splits: the
# prep step may include groups dropped by the 200-project floor, and exclude_groups
# may remove groups that originally cleared the floor. Keep those entries visible
# with kept_for_run=False instead of replacing the list with only surviving groups.
_existing_group_counts = STRATEGIC_RUN_META.get("group_counts") or []
if _existing_group_counts:
    _fresh_count_lookup = {
        str(row[GROUPBY_FIELD]): int(row["project_count"])
        for _, row in _fresh_group_counts.iterrows()
    }
    _refreshed_group_counts = []
    for row in _existing_group_counts:
        group_value = row.get(GROUPBY_FIELD, row.get("strategic_injected_tag", row.get("taxonomy_tag")))
        group_value_key = str(group_value)
        new_row = {**row}
        surviving = group_value_key in _fresh_count_lookup
        new_row["kept_for_run"] = bool(surviving)
        if surviving:
            new_row["project_count"] = int(_fresh_count_lookup[group_value_key])
        _refreshed_group_counts.append(new_row)
    STRATEGIC_RUN_META["group_counts"] = _refreshed_group_counts
else:
    _fresh_records = _fresh_group_counts.to_dict(orient="records")
    for row in _fresh_records:
        row["kept_for_run"] = True
    STRATEGIC_RUN_META["group_counts"] = _fresh_records

# ── Run identity ──────────────────────────────────────────────────────────────
FILTER_SPEC = canonicalize_filter_spec(FILTER_LOGIC, FILTERS)
if STRATEGIC_LOOP_ENABLED:
    RUN_SCOPE_SPEC = deepcopy(FILTER_SPEC)
    RUN_SCOPE_SPEC["strategic_loop"] = {
        "enabled": True,
        "strategic_area_id": STRATEGIC_RUN_META.get("strategic_area_id"),
        "split_id": STRATEGIC_RUN_META.get("split_id"),
        "groupby_field": GROUPBY_FIELD,
    }
else:
    # Preserve v1.4 legacy-mode run hashes by hashing only FILTER_SPEC.
    RUN_SCOPE_SPEC = FILTER_SPEC

FILTER_FIELDS_KEY = get_filter_fields_key(FILTERS)
RUN_DATE = get_run_date()
RUN_ID = get_run_id(GROUPBY_FIELD, RUN_SCOPE_SPEC)

def OUT(subdir, fname):
    return build_run_output_path(
        subdir=subdir,
        fname=fname,
        groupby_field=GROUPBY_FIELD,
        run_date=RUN_DATE,
        run_id=RUN_ID,
    )

# ── Analysis description and topic/slice settings ────────────────────────────
if STRATEGIC_LOOP_ENABLED:
    area_label = STRATEGIC_RUN_META.get("strategic_area_label") or STRATEGIC_RUN_META.get("strategic_area_id")
    split_id = STRATEGIC_RUN_META.get("split_id") or GROUPBY_FIELD
    GROUP_DESCRIPTION = (
        f"strategic area '{area_label}' split by {split_id}; "
        f"group values are values of '{GROUPBY_FIELD}'"
    )
else:
    GROUP_DESCRIPTION = CFG["analysis"]["group_descriptions"].get(
        GROUPBY_FIELD,
        f"groups defined by '{GROUPBY_FIELD}' in DonorsChoose data",
    )

MIN_SHARED    = CFG["analysis"]["nmf_min_shared"]
MIN_COVERAGE  = CFG["analysis"]["min_coverage"]
BASE_N_COMPONENTS = CFG["nmf"]["n_components"]
CAT_TFIDF_TOP_N   = CFG["analysis"]["cat_tfidf_top_n"]

SLICE_RULES   = CFG["analysis"]["slice_rules"]
VERIFY_CFG    = CFG["analysis"]["verification"]
PACKAGING_CFG = CFG["analysis"]["packaging"]
DEDUPE_CFG    = CFG["analysis"]["dedupe"]

group_project_counts = (
    df[[GROUPBY_FIELD, "project_id"]]
    .dropna(subset=["project_id"])
    .drop_duplicates()
    .groupby(GROUPBY_FIELD)["project_id"]
    .nunique()
)
median_group_projects = float(group_project_counts.median()) if not group_project_counts.empty else 0.0
SMALL_SLICE_MODE = median_group_projects < SLICE_RULES["small_slice_cutoff"]
SLICE_RULES = {**SLICE_RULES, "small_slice_mode": SMALL_SLICE_MODE}

MIN_GROUP_PROJECTS_EFFECTIVE = max(
    CFG["analysis"]["min_group_projects"],
    SLICE_RULES["min_group_projects"],
)
CROSS_MIN_INSIGHTS     = SLICE_RULES["cross_min_insights"]
CROSS_MAX_INSIGHTS     = SLICE_RULES["cross_max_insights"]
PER_GROUP_MIN_INSIGHTS = SLICE_RULES["per_group_min_insights"]
PER_GROUP_MAX_INSIGHTS = SLICE_RULES["per_group_max_insights"]

# ── LLM settings ──────────────────────────────────────────────────────────────
N_REPRESENTATIVE          = CFG["llm"]["n_representative_snippets"]
TOP_TERMS_IN_PROMPT       = CFG["llm"]["top_terms_in_prompt"]
SYNTHESIS_TOP_TERMS_COUNT = CFG["llm"]["synthesis_top_terms_count"]
MAX_RETRIES               = CFG["llm"]["max_retries"]

MODEL_LABELING  = CFG["models"]["labeling"]
MODEL_SYNTHESIS = CFG["models"]["synthesis"]
MODEL_VERIFY    = CFG["models"]["verify"]

MAX_WORKERS          = CFG["analysis"]["synthesis_max_workers"]
LABELING_MAX_WORKERS = CFG["analysis"]["labeling_max_workers"]

# ── Metadata lift settings for Step 7 ─────────────────────────────────────────
METADATA_LIFT_CFG = CFG.get("metadata_lift", {}) or {}
METADATA_LIFT_ENABLED = bool(METADATA_LIFT_CFG.get("enabled", True))
if "min_lift" in METADATA_LIFT_CFG:
    print(
        "WARNING: metadata_lift.min_lift is deprecated and ignored. "
        "Use metadata_lift.min_cohens_h instead. Lift is still reported in the prompt "
        "after metadata facts are selected."
    )
METADATA_LIFT_DIMENSIONS = METADATA_LIFT_CFG.get("dimensions", DEFAULT_METADATA_LIFT_DIMENSIONS)
METADATA_LIFT_THRESHOLDS = {
    **DEFAULT_METADATA_LIFT_THRESHOLDS,
    **{k: v for k, v in METADATA_LIFT_CFG.items() if k in DEFAULT_METADATA_LIFT_THRESHOLDS},
}

# ── Output settings ──────────────────────────────────────────────────────────
LOOKER_BASE_URL     = CFG["output"]["looker_base_url"]
LOOKER_FILTER_FIELD = CFG["output"]["looker_filter_field"]
LOOKER_FIELDS       = CFG["output"]["looker_fields"]
LOOKER_LIMIT        = int(CFG["output"].get("looker_limit", 500))
LOOKER_ID_LIMIT     = int(CFG["output"].get("looker_id_limit", 100))
CSV_MAX_IDS_PER_INSIGHT = int(CFG["output"]["csv_max_ids_per_insight"])
MAIN_MIN_VERIFICATION_RATIO = PACKAGING_CFG["main_min_verification_ratio"]

# bins are guarded out above; group_cols is always [GROUPBY_FIELD] in v1.
group_cols = [GROUPBY_FIELD]

# ── Warnings, filter provenance, stage manifest ──────────────────────────────
WARNINGS_PATH       = OUT("metadata", "warnings_03.jsonl")
FILTER_SPEC_PATH    = OUT("metadata", "filter_spec.json")
FILTER_SUMMARY_PATH = OUT("metadata", "filter_summary.json")
STRATEGIC_META_PATH = OUT("metadata", "strategic_run_meta.json")
COPIED_CONFIG_PATH  = OUT("metadata", CFG_PATH.name)

ensure_warning_file(WARNINGS_PATH)

filter_spec_payload = {
    "schema_version": "v1",
    "run_id": RUN_ID,
    "group_by_field": GROUPBY_FIELD,
    "filter_fields_key": FILTER_FIELDS_KEY,
    "filter_logic": FILTER_LOGIC,
    "filters": FILTERS,
}
if STRATEGIC_LOOP_ENABLED:
    filter_spec_payload["strategic_loop"] = RUN_SCOPE_SPEC["strategic_loop"]
write_json(FILTER_SPEC_PATH, filter_spec_payload)

write_json(FILTER_SUMMARY_PATH, {
    "schema_version": "v1",
    "run_id": RUN_ID,
    "group_by_field": GROUPBY_FIELD,
    **filter_summary,
})
write_json(STRATEGIC_META_PATH, {
    "schema_version": "v1",
    "run_id": RUN_ID,
    **STRATEGIC_RUN_META,
})
shutil.copy2(CFG_PATH, COPIED_CONFIG_PATH)

STAGE_MANIFEST = start_stage_manifest(
    stage_name="03_insights_generation",
    notebook_file="03_insights_generation_v1.5.4_strategic_loop.ipynb",
    config_path=MANIFEST_CFG_PATH,
    run_id=RUN_ID,
    group_by_field=GROUPBY_FIELD,
    filter_fields_key=FILTER_FIELDS_KEY,
)

print(f"RUN_ID            = {RUN_ID}")
print(f"GROUPBY_FIELD     = {GROUPBY_FIELD!r}")
print(f"FILTER_FIELDS_KEY = {FILTER_FIELDS_KEY}")
print(f"Filtered projects before strategic prep = {BASE_FILTERED_PROJECT_COUNT:,}")
print(f"Run rows          = {len(df):,}")
print(f"Run projects      = {df['project_id'].nunique():,}")
print(f"Run groups        = {df[GROUPBY_FIELD].nunique(dropna=True):,}")
print(f"small_slice_mode  = {SMALL_SLICE_MODE} (median group = {median_group_projects:.1f})")
print(f"Output root       = OUTPUTS/runs/{GROUPBY_FIELD}/{RUN_DATE}/{RUN_ID}/")

if STRATEGIC_LOOP_ENABLED:
    print("\nStrategic run summary:")
    for key in [
        "strategic_area_id",
        "strategic_area_label",
        "split_id",
        "groupby_fields",
        "is_strategic_injected_tag",
        "min_group_projects",
        "input_project_count",
        "area_project_count",
        "run_project_count",
        "group_count",
        "removed_area_defining_injected_tokens",
    ]:
        print(f"  {key:40s} = {STRATEGIC_RUN_META.get(key)}")


---
## Step 1 — Build TF-IDF Matrices

Fits one TF-IDF vectorizer per n-gram range on the full filtered corpus and
keeps the resulting sparse matrices in memory. All downstream steps reuse these
objects rather than refitting.

Trigrams are built when `ngrams.max_n >= 3` in `params.yaml`.

In [ ]:
docs = df["tokens"].apply(tokens_to_str).tolist()

# Build one matrix per n-gram range. X_unigram_bigram is the primary matrix
# for category TF-IDF and NMF; the others are retained for debugging and
# future analytical passes.
specs = {
    "X_unigram":        (1, 1),
    "X_bigram":         (2, 2),
    "X_unigram_bigram": (1, 2),
}
if CFG["ngrams"]["max_n"] >= 3:
    specs["X_trigram"] = (3, 3)

matrices, vecs = {}, {}
for name, rng in specs.items():
    vec = make_vec(ct["min_df"], ct["max_df"], rng)
    matrices[name] = vec.fit_transform(docs)
    vecs[name] = vec
    sz, nnz = matrices[name].shape, matrices[name].nnz
    print(f"  {name:20s}: shape={sz}  sparsity={1 - nnz / (sz[0] * sz[1]):.3f}")

# Spot-check first features to catch bad filtering or token drift.
for name, vec in vecs.items():
    print(f"  {name:20s} first feats → {vec.get_feature_names_out()[:10].tolist()}")

---
## Step 2 — Quality Checkpoint 2

Gate before LLM calls. Review before proceeding:

- **No stopword violations** — if terms from `quality.stopword_violation_list`
  appear in the top-200 vocab, re-run NB01 with tighter frequency thresholds.
- **Reasonable token distribution** — `p50` should be in the 20–60 range.
- **Matrix sparsity** — very high sparsity on the bigram matrix suggests
  `tfidf.min_df` may be too restrictive.

In [ ]:
# Pass the configured stopword list so the gate reflects params.yaml rather
# than the HARD_STOPWORDS fallback in utils.py.
qr2 = quality_report(
    df, label="cp2",
    matrices=matrices,
    save_path=OUT("quality", "quality_cp2.json"),
    stopwords=STOPWORDS,
)

---
## Step 3 — Category TF-IDF

The vectorizer is fit **once** on the full corpus; category slices are scored
by index. This avoids the string-comparison bug (identical token sets across
projects would misclassify rows) and is much faster than refitting per slice.

**Contrast** = token prevalence in this category minus prevalence outside it.

Time bins: defined in `params.yaml` under `analysis.bins`; leave empty for
the full date range.


In [ ]:
# ── Category TF-IDF ────────────────────────────────────────────────────────
# Score each eligible group against the rest of the corpus using a shared matrix.

top_n = CAT_TFIDF_TOP_N
min_proj = MIN_GROUP_PROJECTS_EFFECTIVE

df_work = df.copy().reset_index(drop=True)
all_docs = df_work["tokens"].apply(tokens_to_str).tolist()

vec_cat = make_vec(ct["min_df"], ct["max_df"], tuple(ct["ngram_range"]))
X_full = vec_cat.fit_transform(all_docs)
X_full = upweight_injected_tokens(
    X_full, vec_cat,
    weight=float(ct.get("injected_token_weight", 1.0)),
    renormalize=True,
)
feat = vec_cat.get_feature_names_out()
idf_vals = vec_cat.idf_

rows = []
for keys, sub in df_work.groupby(group_cols, observed=True):
    if len(sub) < min_proj:
        continue

    kd = group_key(keys, group_cols)
    top = cat_tfidf_slice(
        sub.index,
        df_index=df_work.index,
        X_full=X_full,
        feat=feat,
        idf_vals=idf_vals,
        top_n=top_n,
    )
    for col, val in kd.items():
        top.insert(0, col, val)
    rows.append(top)

if not rows:
    raise RuntimeError(
        f"No groups met min_proj={min_proj} threshold — lower min_group_projects in params.yaml"
    )

cat_tfidf_df = pd.concat(rows, ignore_index=True)
cat_tfidf_df.to_csv(OUT("analysis", "category_tfidf.csv"), index=False)

print(f"{len(cat_tfidf_df):,} rows  |  {cat_tfidf_df[group_cols[0]].nunique()} groups")
cat_tfidf_df.head(10)

---
## Step 4 — NMF Topic Discovery

NMF is fit independently per group so the dominant vocabulary axis in one group
does not suppress signal in others. Topics are treated as evidence candidates
for LLM synthesis — not as stable theme definitions.

The NMF weight matrix `W` records how strongly each project loads on each topic.
Step 5 uses the top-weight projects per topic as representative snippets rather
than sampling randomly.

In [ ]:
# ── Groupwise NMF topics ───────────────────────────────────────────────────
# Fit one NMF model per eligible group and keep both topic-level and project-level outputs.

cn = CFG["nmf"]
min_proj = MIN_GROUP_PROJECTS_EFFECTIVE

_itw = float(ct.get("injected_token_weight", 1.0))
if _itw != 1.0:
    print(f"  injected_token_weight = {_itw} (L2-renormalized per row)")

all_topics, all_weights = [], []
df_work = df.copy().reset_index(drop=True)

NMF_GROUPS_SKIPPED = []
NMF_GROUPS_FAILED = []

for keys, sub in df_work.groupby(group_cols, observed=True):
    kd = group_key(keys, group_cols)
    group_value = kd[GROUPBY_FIELD]

    # Skip groups that are too small to support stable topic extraction.
    if len(sub) < min_proj:
        NMF_GROUPS_SKIPPED.append(str(group_value))
        continue

    group_docs = sub["tokens"].apply(tokens_to_str).tolist()
    pids = sub["project_id"].tolist()

    try:
        topics, W, nmf_meta = nmf_one(
            group_docs,
            ct_cfg=ct,
            cn_cfg=cn,
            base_n_components=BASE_N_COMPONENTS,
            slice_rules=SLICE_RULES,
        )
    except Exception as e:
        NMF_GROUPS_FAILED.append(str(group_value))
        append_warning(
            WARNINGS_PATH,
            "03_insights_generation",
            "NMF_GROUP_SKIPPED",
            f"NMF failed for group '{group_value}'",
            context={"group": group_value, "error": str(e)},
        )
        continue

    # None return means the slice was too thin; kept separate from exceptions for QA.
    if topics is None or W is None:
        NMF_GROUPS_FAILED.append(str(group_value))
        append_warning(
            WARNINGS_PATH,
            "03_insights_generation",
            "NMF_GROUP_SKIPPED",
            f"NMF skipped group '{group_value}' because the slice was too thin",
            context={"group": group_value, **(nmf_meta or {})},
        )
        continue

    for col, val in kd.items():
        topics[col] = val
    all_topics.append(topics)

    # Persist the full W ranking so later cells can recover top projects per topic.
    for tid in range(W.shape[1]):
        order = W[:, tid].argsort()[::-1]
        for rank, idx in enumerate(order):
            all_weights.append(
                {
                    **kd,
                    "topic_id": tid,
                    "project_id": pids[idx],
                    "weight": float(W[idx, tid]),
                    "rank": rank,
                }
            )

if not all_topics:
    raise RuntimeError(
        "No groups produced NMF topics — check n_components vs retained vocab, "
        "or lower min_group_projects"
    )

topics_df = pd.concat(all_topics, ignore_index=True)
weights_df = pd.DataFrame(all_weights)

topics_df.to_csv(OUT("analysis", "nmf_topics.csv"), index=False)
weights_df.to_csv(OUT("analysis", "nmf_weights.csv"), index=False)

print(f"{len(topics_df):,} topics across {topics_df[group_cols[0]].nunique()} groups")

# Build the project-topic bridge once so downstream evidence collection can reuse it.
threshold = CFG["analysis"]["topic_assignment_threshold"]
project_topic_bridge_df = build_project_topic_bridge(
    weights_df,
    GROUPBY_FIELD,
    threshold,
)
project_topic_bridge_df.to_csv(OUT("analysis", "project_topic_bridge.csv"), index=False)
print(
    "Bridge: "
    f"{len(project_topic_bridge_df):,} project-topic assignments "
    f"(topic_share >= {threshold})"
)

topics_df.head(6)

In [ ]:
# ── Cross-group universal themes ───────────────────────────────────────────
# Identify term bundles that recur across groups after topic extraction.

# Select the largest group as the cross-group reference point for any
# group-specific context needed in downstream steps.
REFERENCE_GROUP = df_work.groupby(GROUPBY_FIELD).size().idxmax()

records = [(r[group_cols[0]], frozenset(r.top_terms)) for _, r in topics_df.iterrows()]

# theme_cats maps a shared term bundle to the set of groups where it recurs.
theme_cats = defaultdict(set)
for i, (ci, si) in enumerate(records):
    for cj, sj in records[i + 1:]:
        if ci == cj:
            continue
        shared = si & sj
        if len(shared) >= MIN_SHARED:
            key = frozenset(shared)
            theme_cats[key] |= {ci, cj}

rows = sorted(theme_cats.items(), key=lambda x: -len(x[1]))
seen = []
deduped = []
for terms, cats in rows:
    if len(cats) < MIN_COVERAGE:
        continue
    if not any(terms <= prior for prior in seen):
        deduped.append(
            {
                "theme": ", ".join(sorted(terms)[:5]),
                "n_groups": len(cats),
                "categories": sorted(cats),
            }
        )
        seen.append(terms)

cross_group_df = pd.DataFrame(deduped).reset_index(drop=True)
cross_group_df.to_csv(OUT("analysis", "cross_group_themes.csv"), index=False)
cross_group_df

In [ ]:
# ── Topic clustering layer (additive, behind topic_clustering.enabled flag) ──
# Builds analysis units between Step 4 NMF topics and Step 5 LLM labeling.
# Produces:
#   analysis_units_df       — rows are clusters + singletons; this is the new
#                             input shape for Step 5 when enabled
#   cluster_membership_df   — per-cluster member rows with group, topic_id,
#                             distinctives, and cluster_support_score
#   generic_clusters_df     — clusters that failed the concrete-core rule;
#                             saved for audit even when dropped from units
#
# When topic_clustering.enabled is False the cell still runs and writes
# diagnostic CSVs, but does not replace topics_df. Step 5 should branch on
# topic_clustering.enabled.

from collections import Counter, defaultdict
from itertools import combinations

TC_CFG = CFG.get("topic_clustering", {}) or {}
TC_ENABLED          = bool(TC_CFG.get("enabled", False))
TC_JACCARD          = float(TC_CFG.get("jaccard_threshold", 0.50))
TC_TOP_N            = int(TC_CFG.get("top_n_terms", 10))
TC_MIN_CONCRETE     = int(TC_CFG.get("min_concrete_core_terms", 3))
TC_GENERIC_TERMS    = set(TC_CFG.get("generic_core_terms", []) or [])
TC_ACTION           = str(TC_CFG.get("generic_cluster_action", "drop")).lower()
assert TC_ACTION in {"drop", "demand_concrete", "pass_through"}, (
    f"topic_clustering.generic_cluster_action must be drop|demand_concrete|pass_through, got {TC_ACTION!r}"
)

# 1. Build topic records and pairwise adjacency at the configured threshold ───
topic_records = [
    {"idx": i,
     "group": row[GROUPBY_FIELD],
     "topic_id": int(row["topic_id"]),
     "top_terms": list(row["top_terms"][:TC_TOP_N])}
    for i, (_, row) in enumerate(topics_df.iterrows())
]
sigs = [frozenset(t["top_terms"]) for t in topic_records]
pairwise_jaccard = {}  # (i,j) -> jaccard, for medoid tiebreak reuse
adj = defaultdict(set)
for i, j in combinations(range(len(topic_records)), 2):
    if topic_records[i]["group"] == topic_records[j]["group"]:
        continue
    inter = len(sigs[i] & sigs[j])
    if not inter:
        continue
    jac = inter / len(sigs[i] | sigs[j])
    pairwise_jaccard[(i, j)] = jac
    pairwise_jaccard[(j, i)] = jac
    if jac >= TC_JACCARD:
        adj[i].add(j); adj[j].add(i)

# 2. Connected components -> clusters ─────────────────────────────────────────
seen, clusters = set(), []
for i in range(len(topic_records)):
    if i in seen or i not in adj:
        continue
    stack, comp = [i], []
    while stack:
        n = stack.pop()
        if n in seen:
            continue
        seen.add(n); comp.append(n)
        stack.extend(adj[n] - seen)
    if len(comp) >= 2:
        clusters.append(sorted(comp))

clustered_idxs = {k for c in clusters for k in c}
singleton_idxs = [i for i in range(len(topic_records)) if i not in clustered_idxs]

# 3. Per-cluster: shared core, distinctives, medoid, project support ──────────
# Project support: join project_topic_bridge_df, sum topic_share per project
# across cluster members, dedup, name the result cluster_support_score.
ptb = project_topic_bridge_df  # from Step 4
ptb_indexed = ptb.set_index([GROUPBY_FIELD, "topic_id"]) if not ptb.empty else None

cluster_rows = []
membership_rows = []
generic_rows = []
units_rows = []

for cid, comp in enumerate(clusters):
    members = [topic_records[k] for k in comp]
    term_sets = [set(m["top_terms"]) for m in members]
    union_terms = set().union(*term_sets)
    inter_terms = set.intersection(*term_sets) if term_sets else set()
    counts = Counter(t for s in term_sets for t in s)
    # shared_core = terms present in at least half of the members (and >=2)
    threshold_count = max(2, math.ceil(len(members) / 2))
    shared_core = [t for t, c in counts.most_common() if c >= threshold_count]
    concrete_core = [t for t in shared_core if t not in TC_GENERIC_TERMS]
    is_generic = len(concrete_core) < TC_MIN_CONCRETE

    # Per-member distinctives: terms unique to this member within the cluster
    per_member_distinct = {}
    for m in members:
        others_union = set().union(
            *[set(mm["top_terms"]) for mm in members if mm["idx"] != m["idx"]]
        )
        per_member_distinct[m["idx"]] = sorted(set(m["top_terms"]) - others_union)

    # Medoid: highest mean Jaccard to other members. Tiebreak: supporting
    # project count from project_topic_bridge_df. Final fallback: topic_id.
    def _mean_jac(i):
        others = [j for j in [mm["idx"] for mm in members] if j != i]
        if not others:
            return 0.0
        return sum(pairwise_jaccard.get((i, j), 0.0) for j in others) / len(others)

    def _support_count(m):
        if ptb_indexed is None:
            return 0
        key = (m["group"], m["topic_id"])
        try:
            sub = ptb_indexed.loc[[key]]
        except KeyError:
            return 0
        return int(sub["project_id"].nunique())

    member_scores = sorted(
        members,
        key=lambda m: (-_mean_jac(m["idx"]), -_support_count(m), m["topic_id"]),
    )
    medoid = member_scores[0]

    # Cluster support: union of member project rows, summed topic_share
    if ptb_indexed is not None:
        member_keys = [(m["group"], m["topic_id"]) for m in members]
        member_keys_in_ptb = [k for k in member_keys if k in ptb_indexed.index]
        if member_keys_in_ptb:
            sup = ptb_indexed.loc[member_keys_in_ptb].reset_index()
            support_by_pid = (sup.groupby("project_id", as_index=False)
                                 .agg(cluster_support_score=("topic_share", "sum"),
                                      n_member_topics=("topic_id", "nunique")))
            support_by_pid = support_by_pid.sort_values("cluster_support_score", ascending=False)
            n_supporting = len(support_by_pid)
            top_supporters = support_by_pid["project_id"].head(50).tolist()
        else:
            support_by_pid = pd.DataFrame(columns=["project_id", "cluster_support_score", "n_member_topics"])
            n_supporting = 0
            top_supporters = []
    else:
        support_by_pid = pd.DataFrame(columns=["project_id", "cluster_support_score", "n_member_topics"])
        n_supporting = 0
        top_supporters = []

    groups_present = sorted({m["group"] for m in members})
    variation_ratio = round(
        (len(union_terms) - len(inter_terms)) / max(len(union_terms), 1), 3
    )

    cluster_row = {
        "cluster_id": cid,
        "n_topics": len(members),
        "n_groups": len(groups_present),
        "shared_core": shared_core,
        "concrete_shared_core": concrete_core,
        "n_shared": len(shared_core),
        "n_concrete_shared": len(concrete_core),
        "variation_ratio": variation_ratio,
        "is_generic": is_generic,
        "medoid_group": medoid["group"],
        "medoid_topic_id": medoid["topic_id"],
        "medoid_top_terms": medoid["top_terms"],
        "groups_present": groups_present,
        "n_supporting_projects": n_supporting,
        "top_supporting_project_ids": top_supporters,
    }
    cluster_rows.append(cluster_row)

    for m in members:
        membership_rows.append({
            "cluster_id": cid,
            GROUPBY_FIELD: m["group"],
            "topic_id": m["topic_id"],
            "is_medoid": (m["idx"] == medoid["idx"]),
            "distinctive_terms": per_member_distinct[m["idx"]],
            "top_terms": m["top_terms"],
        })

    if is_generic:
        generic_rows.append(cluster_row)
        if TC_ACTION == "drop":
            continue
        # demand_concrete or pass_through still emits a unit; downstream prompt
        # branch is responsible for handling is_generic=True appropriately.

    units_rows.append({
        "unit_id": f"cluster_{cid}",
        "unit_type": "cluster",
        "cluster_id": cid,
        "group": None,
        "topic_id": None,
        "is_generic": is_generic,
        "n_topics": len(members),
        "n_groups": len(groups_present),
        "n_supporting_projects": n_supporting,
        "shared_core": shared_core,
        "concrete_shared_core": concrete_core,
        "variation_ratio": variation_ratio,
        "medoid_top_terms": medoid["top_terms"],
        "groups_present": groups_present,
    })

# 4. Singleton units ──────────────────────────────────────────────────────────
for i in singleton_idxs:
    t = topic_records[i]
    n_sup = 0
    if ptb_indexed is not None:
        try:
            n_sup = int(ptb_indexed.loc[[(t["group"], t["topic_id"])]]["project_id"].nunique())
        except KeyError:
            n_sup = 0
    units_rows.append({
        "unit_id": f"singleton_{t['group']}_{t['topic_id']}",
        "unit_type": "singleton",
        "cluster_id": None,
        "group": t["group"],
        "topic_id": t["topic_id"],
        "is_generic": False,
        "n_topics": 1,
        "n_groups": 1,
        "n_supporting_projects": n_sup,
        "shared_core": None,
        "concrete_shared_core": None,
        "variation_ratio": None,
        "medoid_top_terms": t["top_terms"],
        "groups_present": [t["group"]],
    })

analysis_units_df = pd.DataFrame(units_rows)
cluster_membership_df = pd.DataFrame(membership_rows)
generic_clusters_df = pd.DataFrame(generic_rows)

# 5. Summary printout ─────────────────────────────────────────────────────────
n_clusters_kept = analysis_units_df.query("unit_type == 'cluster'").shape[0]
n_singletons    = analysis_units_df.query("unit_type == 'singleton'").shape[0]
n_generic_drop  = len(generic_clusters_df) if TC_ACTION == "drop" else 0

print("─" * 72)
print("TOPIC CLUSTERING LAYER")
print("─" * 72)
print(f"  enabled                        = {TC_ENABLED}")
print(f"  input topics                   = {len(topic_records)}")
print(f"  clusters kept as units         = {n_clusters_kept}")
print(f"  singletons                     = {n_singletons}")

# 6. Save artifacts ───────────────────────────────────────────────────────────
analysis_units_df.to_csv(OUT("analysis", "analysis_units.csv"), index=False)
cluster_membership_df.to_csv(OUT("analysis", "cluster_membership.csv"), index=False)
generic_clusters_df.to_csv(OUT("analysis", "generic_clusters.csv"), index=False)


---
## Step 5 — LLM Topic Labeling

One API call per topic using compressed input — never raw essay text at scale.
Representative snippets are selected by NMF weight (highest-loading projects),
not randomly. Parallel execution via `ThreadPoolExecutor`.

Parse failures are stored with enough metadata to debug later; failed topics are
excluded from synthesis but do not halt the run.

In [ ]:
# ── LLM topic labeling (forks on topic_clustering.enabled) ────────────────────
# When topic_clustering.enabled is True, dispatches per analysis_units_df row:
#   unit_type == "cluster"   → cluster prompt, _label_cluster_with_retry
#   unit_type == "singleton" → original topic prompt, _label_with_retry
# When False, runs the legacy per-topic path against topics_df unchanged.
#
# Outputs:
#   results              — all labels (cluster + singleton) in unit_order
#   unit_labels_df       — DataFrame of all successful labels with unit_type
#   labels_df            — legacy-compatible singleton-only DataFrame
#                          (also includes singletons-from-topics in legacy mode)
#   llm_topic_labels.json — full results, ordered

# Read topic_clustering config from either top-level or under analysis.
TC_CFG = (
    CFG.get("topic_clustering")
    or CFG.get("analysis", {}).get("topic_clustering")
    or {}
)
TC_ENABLED = bool(TC_CFG.get("enabled", False))

# ── Topic prompt (singleton path + legacy path) ───────────────────────────────
SYSTEM = (
    "You are an NLP analyst reviewing NMF topic clusters from DonorsChoose teacher "
    "essays. Respond ONLY with a single valid JSON object. No preamble. No markdown fences.\n\n"

    "Input characteristics:\n"
    "- All tokens shown have been preprocessed: lowercased, lemmatized, deduplicated within "
    "each project, and filtered against a corpus-wide stopword list. Common stopwords "
    "('the', 'and', 'for', 'this', 'student', 'classroom', 'project', 'school', 'teacher', etc.) "
    "and very-rare terms have been removed before TF-IDF.\n"
    "- The 'Top NMF terms' are the highest-weighted terms NMF assigned to this topic and "
    "are the strongest single evidence of the topic's content.\n"
    "- The 'Representative project tokens' are token-level snippets from the highest-loading "
    "projects, not raw essay prose. Do not quote them as if they were sentences. Do not "
    "infer narrative or rhetorical style from token order or co-occurrence within a snippet.\n\n"
    "Token naming conventions you may see in topic terms:\n"
    "  __framing_[name]__ = rhetorical framing, tone, mechanism, or persuasion signal\n"
    "  __subject_[name]__ = subject/domain/content signal\n"
    "  __industry_[name]__ = workforce-development industry or skill-domain signal\n"
    "  __request_[name]__ = material/request-topology signal\n"
    "  __context_[name]__ = contextual school, community, attendance, safety, or access signal\n"
    "  __sensitive_context_[name]__ = direct sensitive-context signal; describe carefully and only when supported\n"
    "  __cat_[name]__ = legacy subject matter category token\n"
    "  __sub_[name]__ = legacy subcategory token\n"
    "These injected tokens are analyst-curated semantic signals. When they appear alongside compatible "
    "terms or snippets, treat them as important evidence about the topic's function, context, or framing, "
    "not as incidental noise. They may help distinguish superficially similar topics. However, do not use "
    "an injected token as standalone proof, and do not let it override direct contradictions in the concrete "
    "terms or representative snippets.\n"
    "Never output raw tokens such as __framing_*__, __subject_*__, __industry_*__, __request_*__, "
    "__context_*__, __sensitive_context_*__, __cat_*__, __sub_*__, or snake_case token names. "
    "Always translate such signals into plain English.\n\n"

    "Language constraints:\n"
    "- Do not use em dashes.\n"
    "- Do not use reveal-style phrasing such as 'not just X,' 'really about Y,' or 'what looks like X is Y.'\n"
    "- Do not use named places.\n\n"

    "Your job is to produce a stable canonical topic label and one dense grounded description.\n"
    "Do not optimize for cleverness, vividness, or donor-facing insight language. "
    "Optimize for specificity, repeatability, plain-English clarity, and preservation of useful evidence.\n\n"

    "Rules for proposed_label:\n"
    "- proposed_label must be a short canonical noun phrase, usually 3 to 7 words.\n"
    "- Prefer the most specific defensible mechanism, request type, intervention, "
    "classroom routine, or use case.\n"
    "- Do not try to pack every nuance into proposed_label.\n"
    "- Do not collapse topics into broad umbrella labels like technology, literacy, "
    "engagement, classroom supplies, or social-emotional learning if the evidence "
    "supports a narrower interpretation.\n"
    "- Preserve concrete signals such as named programs, pedagogies, student populations, "
    "classroom routines, and rhetorical framing when clearly supported.\n\n"

    "Rules for description:\n"
    "- description must be exactly one sentence.\n"
    "- description must be plain English and evidence-led.\n"
    "- Write the description as one grounded sentence naming the dominant evidence pattern: "
    "the concrete materials or activities, the classroom function when supported, and the population, "
    "setting, or framing signal only when clearly present.\n"
    "- Prefer concrete natural-language phrasing over abstract wording.\n"
    "- Do not write implications, recommendations, or donor-facing interpretation.\n"
    "- Do not use raw preprocessing tokens or token-like jargon in description.\n\n"

    "Rules for notes:\n"
    "- notes should usually be empty.\n"
    "- Use notes only for real edge cases.\n"
    "- If coherence_flag is mixed, briefly name the colliding subthemes.\n"
    "- If coherence_flag is redundant, briefly state the nature of the overlap.\n"
    "- If coherence_flag is unclear, briefly state why the topic is too scattered.\n"
    "- Do not restate the description in notes.\n\n"

    "coherence_flag definitions:\n"
    "  coherent   — one dominant theme, mechanism, population, or use case clearly leads, "
    "even if secondary signals are present.\n"
    "  mixed      — no single dominant theme clearly leads and two or more distinguishable "
    "subthemes are truly colliding.\n"
    "  redundant  — this topic's terms and snippets substantially duplicate another "
    "topic in the same group, not merely overlap in subject area.\n"
    "  unclear    — terms are too scattered or generic to support a defensible label.\n\n"

    "Important tie-breaker:\n"
    "- If one theme is primary and another is secondary, mark the topic coherent, not mixed, "
    "and keep notes empty unless the secondary signal is important to preserve.\n"
    "- If two phrasings are both plausible, choose the more literal, reusable, and plain-English one."
)

PROMPT = (
    f"Group field: {GROUPBY_FIELD} — {GROUP_DESCRIPTION}\n"
    f"Group value: {{group_value}}{{bin_line}}\n"
    "Topic {topic_id}\n"
    "Top unigrams : {unigrams}\n"
    "Top bigrams  : {bigrams}\n"
    "Top NMF terms: {nmf_terms}\n"
    "Representative project tokens:\n"
    "{snippets}\n\n"

    "Instructions:\n"
    "- proposed_label must be a short canonical noun phrase, usually 3 to 7 words.\n"
    "- description must be exactly one sentence in plain English.\n"
    "- Write the description as one grounded sentence naming the dominant evidence pattern: "
    "the concrete materials or activities, the classroom function when supported, and the population, "
    "setting, or framing signal only when clearly present.\n"
    "- Prefer concrete mechanism and classroom function over broad category language.\n"
    "- Translate any framing/category/subcategory token signals into normal language; never copy raw token strings.\n"
    "- If the topic appears broad, identify the narrower mechanism, use case, or population "
    "if the evidence supports it.\n"
    "- Use coherence_flag='mixed' only when no single dominant theme clearly leads.\n"
    "- If one theme is primary and another is secondary, mark the topic coherent.\n"
    "- notes should usually be empty; use notes only for mixed, redundant, or unclear topics.\n"
    "- Do not write donor-facing insights, implications, or rhetorical flourishes.\n\n"

    "Output requirements:\n"
    "- proposed_label: short and stable\n"
    "- description: exactly one dense grounded sentence\n"
    '- coherence_flag: one of "coherent", "mixed", "redundant", "unclear"\n'
    "- notes: usually empty, unless needed for edge cases\n\n"

    f"Return a JSON object with exactly these keys:\n"
    f"{GROUPBY_FIELD}, topic_id, proposed_label, description, coherence_flag, notes"
)

# ── Cluster prompt (cluster path only) ────────────────────────────────────────
CLUSTER_SYSTEM = (
    "You are an NLP analyst labeling cross-group NMF topic clusters from DonorsChoose "
    "teacher essays. A cluster is a recurring classroom-need family: multiple NMF topics "
    "from different group slices that share a core of terms. Your job is to label the "
    "shared family and capture how it varies across the groups that contain it.\n\n"

    "Respond ONLY with a single valid JSON object. No preamble. No markdown fences.\n\n"

    "Input characteristics:\n"
    "- All tokens shown have been preprocessed: lowercased, lemmatized, deduplicated within "
    "each project, and filtered against a corpus-wide stopword list. Common stopwords "
    "('the', 'and', 'for', 'this', 'student', 'classroom', 'project', 'school', 'teacher', etc.) "
    "and very-rare terms have been removed before TF-IDF.\n"
    "- 'Shared core' lists terms present in at least half of the cluster's member topics' top "
    "terms; treat these as the cluster's identity.\n"
    "- Each member's 'Distinctive terms' are terms in only that member's top terms among the "
    "cluster. These are the strongest evidence for variation_notes.\n"
    "- The 'Representative project tokens' lines are token-level snippets from the highest-loading "
    "project per member group, not raw essay prose. Do not quote them as sentences.\n\n"

    "Token naming conventions you may see in cluster terms:\n"
    "  __framing_[name]__ = rhetorical framing or tone signal\n"
    "  __subject_[name]__ = subject/domain signal\n"
    "  __industry_[name]__ = workforce-development industry signal\n"
    "  __request_[name]__ = material/request-topology signal\n"
    "  __context_[name]__ = contextual school/community signal\n"
    "  __sensitive_context_[name]__ = direct sensitive-context signal\n"
    "  __cat_[name]__ = legacy category token\n"
    "  __sub_[name]__ = legacy subcategory token\n"
    "Translate these into plain English; never output raw token strings.\n\n"

    "Language constraints:\n"
    "- Do not use em dashes.\n"
    "- Do not use reveal-style phrasing.\n"
    "- Do not use named places.\n\n"

    "Rules for proposed_label:\n"
    "- proposed_label is a short canonical noun phrase, usually 3 to 7 words.\n"
    "- Name the recurring classroom-need family, not any single member's specific angle.\n"
    "- Prefer concrete request/mechanism language over abstract umbrella terms.\n"
    "- Do not pack every member's distinctive into the label.\n\n"

    "Rules for description:\n"
    "- description must be exactly one sentence in plain English.\n"
    "- Describe what the cluster's shared core represents as a recurring need pattern.\n"
    "- Do not enumerate per-group differences in description; that is what variation_notes is for.\n"
    "- Do not write donor-facing insights, implications, or recommendations.\n\n"

    "Rules for variation_notes:\n"
    "- variation_notes is a list with exactly one entry per member group listed in 'Groups present'.\n"
    "- Each entry has two fields: 'group' (exact group value from Groups present) and "
    "'distinctive_angle' (plain-English description of what makes this member's contribution distinctive).\n"
    "- distinctive_angle should reference that member's distinctive terms and top NMF terms.\n"
    "- If a member has no meaningful distinctive angle beyond the shared core, set distinctive_angle "
    "to exactly 'no distinctive angle'.\n"
    "- Do not invent groups not in Groups present. Do not omit any group in Groups present. "
    "Do not list the same group more than once.\n\n"

    "Rules for coherence_flag:\n"
    "  coherent  — one shared family clearly leads, variation across groups is consistent and complementary.\n"
    "  mixed     — the shared core is real but variation notes describe genuinely incompatible angles.\n"
    "  redundant — the cluster has no concrete variation; every member says roughly the same thing.\n"
    "  unclear   — cannot tell what binds the cluster.\n\n"

    "Rules for notes:\n"
    "- notes should usually be empty.\n"
    "- Use notes only for real edge cases.\n"
    "- If coherence_flag is mixed, briefly name the colliding angles.\n"
    "- If coherence_flag is redundant, briefly say so.\n"
    "- If coherence_flag is unclear, briefly state why.\n"
    "- Do not restate description in notes."
)

CLUSTER_PROMPT = (
    f"Group field: {GROUPBY_FIELD} — {GROUP_DESCRIPTION}\n\n"
    "Cluster {cluster_id}\n"
    "Members          : {n_topics} topics across {n_groups} groups\n"
    "Shared core      : {shared_core}\n"
    "Concrete core    : {concrete_shared_core}\n"
    "Medoid group     : {medoid_group}\n"
    "Medoid top terms : {medoid_top_terms}\n"
    "Groups present   : {groups_present}\n\n"
    "Per-member detail:\n"
    "{members_block}\n\n"

    "Output requirements:\n"
    "- proposed_label : short noun phrase naming the shared family\n"
    "- description    : one dense grounded sentence about the shared core\n"
    "- variation_notes: list with one entry per group in Groups present, each appearing exactly once\n"
    "- coherence_flag : one of 'coherent', 'mixed', 'redundant', 'unclear'\n"
    "- notes          : usually empty\n\n"

    "Return a JSON object with exactly these keys: "
    "cluster_id, proposed_label, description, variation_notes, coherence_flag, notes"
)

# ── pid_text snippet lookup (dedup before set_index to handle exploded rows) ──
pid_text = (
    df.drop_duplicates("project_id")
      .set_index("project_id")["tokens"]
      .apply(lambda t: " ".join(coerce_token_list(t)[:40]))
)

# ── Dispatch ──────────────────────────────────────────────────────────────────
if TC_ENABLED:
    if "analysis_units_df" not in dir() or analysis_units_df.empty:
        raise RuntimeError(
            "topic_clustering.enabled is True but analysis_units_df is missing or empty. "
            "Re-run the topic clustering layer cell before Step 5."
        )

    cluster_units = analysis_units_df[analysis_units_df["unit_type"] == "cluster"]
    singleton_units = analysis_units_df[analysis_units_df["unit_type"] == "singleton"]
    print(f"Step 5 dispatch: {len(cluster_units)} cluster(s), {len(singleton_units)} singleton(s).")

    # Stable unit ordering so the output JSON is deterministic across runs.
    unit_order = {
        str(row["unit_id"]): i
        for i, (_, row) in enumerate(analysis_units_df.iterrows())
    }

    # Build cluster inputs.
    cluster_inputs = [
        build_cluster_input(
            row,
            membership_df=cluster_membership_df,
            weights_df=weights_df,
            pid_text=pid_text,
            groupby_field=GROUPBY_FIELD,
            n_representative_per_member=1,
            top_terms_per_member=8,
        )
        for _, row in cluster_units.iterrows()
    ]

    # Build singleton inputs by reconstructing a topic-row shape per unit.
    # Carry the precomputed unit_id from analysis_units_df so we never
    # synthesize unsafe IDs from raw group values.
    singleton_inputs = []
    for _, srow in singleton_units.iterrows():
        topic_row = topics_df[
            (topics_df[GROUPBY_FIELD] == srow["group"])
            & (topics_df["topic_id"] == srow["topic_id"])
        ]
        if topic_row.empty:
            append_warning(
                WARNINGS_PATH, "03_insights_generation", "SINGLETON_TOPIC_MISSING",
                f"Singleton unit references topic not in topics_df: "
                f"{srow['group']} / {srow['topic_id']}",
                context={"group": str(srow["group"]), "topic_id": int(srow["topic_id"])},
            )
            continue
        inp = build_input(
            topic_row.iloc[0],
            weights_df=weights_df,
            pid_text=pid_text,
            groupby_field=GROUPBY_FIELD,
            n_representative=N_REPRESENTATIVE,
            top_terms_in_prompt=TOP_TERMS_IN_PROMPT,
        )
        inp["unit_id"] = str(srow["unit_id"])
        singleton_inputs.append(inp)

    # ── Sanity checks before paid LLM calls ──────────────────────────────────
    if cluster_inputs:
        sample_cluster = cluster_inputs[0]
        if isinstance(sample_cluster.get("shared_core"), str):
            raise TypeError(
                "Cluster shared_core is still a string after coercion. "
                "Check coerce_token_list() before running LLM calls."
            )
        if sample_cluster.get("members"):
            sample_member = sample_cluster["members"][0]
            if isinstance(sample_member.get("top_terms"), str):
                raise TypeError(
                    "Cluster member top_terms is still a string after coercion. "
                    "Check coerce_token_list() before running LLM calls."
                )

    if singleton_inputs:
        sample_singleton = singleton_inputs[0]
        if not sample_singleton.get("group_value") or sample_singleton.get("topic_id") is None:
            raise ValueError(
                "Singleton input is missing group_value or topic_id. "
                "Check singleton unit reconstruction before running LLM calls."
            )
    
    results = []
    with ThreadPoolExecutor(max_workers=LABELING_MAX_WORKERS) as executor:
        futures = {}
        for inp in cluster_inputs:
            futures[executor.submit(
                _label_cluster_with_retry,
                inp,
                client=client,
                model_labeling=MODEL_LABELING,
                system_prompt=CLUSTER_SYSTEM,
                user_prompt_template=CLUSTER_PROMPT,
                warnings_path=WARNINGS_PATH,
                max_retries=MAX_RETRIES,
            )] = ("cluster", inp)
        for inp in singleton_inputs:
            futures[executor.submit(
                _label_with_retry,
                inp,
                client=client,
                model_labeling=MODEL_LABELING,
                system_prompt=SYSTEM,
                user_prompt_template=PROMPT,
                groupby_field=GROUPBY_FIELD,
                warnings_path=WARNINGS_PATH,
                max_retries=MAX_RETRIES,
            )] = ("singleton", inp)
        for future in as_completed(futures):
            kind, inp = futures[future]
            obj = future.result()
            if kind == "singleton":
                obj["unit_type"] = "singleton"
                obj["unit_id"] = inp["unit_id"]
            results.append(obj)
            disp = obj.get("proposed_label", "?")
            tag = (
                f"cluster {inp['cluster_id']}"
                if kind == "cluster"
                else f"{inp['group_value']} / topic {inp['topic_id']}"
            )
            print(f"  [{kind:9s}] {tag} → {disp}")

    # Sort results into stable unit order.
    results = sorted(
        results,
        key=lambda r: unit_order.get(str(r.get("unit_id", "")), 10**9),
    )

    with open(OUT("analysis", "llm_topic_labels.json"), "w", encoding="utf-8") as f:
        json.dump(results, f, indent=2, ensure_ascii=False, default=str)

    # ── Build two DataFrames so downstream can choose its shape ──────────────
    # unit_labels_df: all successful labels with unit_type column.
    # labels_df:      legacy-compatible shape (singletons only, with
    #                 GROUPBY_FIELD and topic_id), so existing helpers like
    #                 build_topic_lines() keep working until they are
    #                 explicitly upgraded for cluster-aware Step 6.
    successful = [r for r in results if not r.get("parse_error")]
    unit_labels_df = pd.DataFrame(successful)

    singleton_label_rows = []
    for r in successful:
        if r.get("unit_type") != "singleton":
            continue
        # Recover GROUPBY_FIELD and topic_id from the matching singleton_unit row.
        u = singleton_units[singleton_units["unit_id"] == r.get("unit_id")]
        if u.empty:
            continue
        srow = u.iloc[0]
        legacy_row = {
            GROUPBY_FIELD: str(srow["group"]),
            "topic_id": int(srow["topic_id"]),
            "proposed_label": r.get("proposed_label"),
            "description": r.get("description"),
            "coherence_flag": r.get("coherence_flag"),
            "notes": r.get("notes"),
            "model": r.get("model"),
            "timestamp": r.get("timestamp"),
            "unit_id": r.get("unit_id"),
            "unit_type": "singleton",
        }
        singleton_label_rows.append(legacy_row)
    labels_df = pd.DataFrame(singleton_label_rows)

    n_clusters_ok = sum(1 for r in successful if r.get("unit_type") == "cluster")
    n_singletons_ok = sum(1 for r in successful if r.get("unit_type") == "singleton")
    n_errors = len(results) - len(successful)
    n_validation_warn = sum(1 for r in successful if r.get("validation_warning"))
    print(f"\n{len(results)} labels saved  "
          f"(clusters ok: {n_clusters_ok}, singletons ok: {n_singletons_ok}, "
          f"errors: {n_errors}, validation warnings: {n_validation_warn})")
    print(f"unit_labels_df: {len(unit_labels_df)} rows  |  "
          f"labels_df (legacy singleton shape): {len(labels_df)} rows")

else:
    # ── Legacy path: per-topic labeling against topics_df, unchanged ──────────
    topic_inputs = [
        build_input(
            t,
            weights_df=weights_df,
            pid_text=pid_text,
            groupby_field=GROUPBY_FIELD,
            n_representative=N_REPRESENTATIVE,
            top_terms_in_prompt=TOP_TERMS_IN_PROMPT,
        )
        for _, t in topics_df.iterrows()
    ]

    topic_order = {
        (_norm_group_value(inp["group_value"]), int(inp["topic_id"])): i
        for i, inp in enumerate(topic_inputs)
    }

    results = []
    with ThreadPoolExecutor(max_workers=LABELING_MAX_WORKERS) as executor:
        futures = {
            executor.submit(
                _label_with_retry,
                inp,
                client=client,
                model_labeling=MODEL_LABELING,
                system_prompt=SYSTEM,
                user_prompt_template=PROMPT,
                groupby_field=GROUPBY_FIELD,
                warnings_path=WARNINGS_PATH,
                max_retries=MAX_RETRIES,
            ): inp
            for inp in topic_inputs
        }
        for future in as_completed(futures):
            inp = futures[future]
            obj = future.result()
            results.append(obj)
            print(f"  {inp['group_value']} / topic {inp['topic_id']} → {obj.get('proposed_label', '?')}")

    bad_sort_keys = []
    for r in results:
        norm_key = (
            _norm_group_value(r.get(GROUPBY_FIELD, "")),
            _safe_topic_id(r.get("topic_id", -1)),
        )
        if norm_key not in topic_order:
            bad_sort_keys.append({"group": r.get(GROUPBY_FIELD, ""), "topic_id": r.get("topic_id", "")})

    if bad_sort_keys:
        raise ValueError(
            f"LABELING_SORT_KEY_MISMATCH: {len(bad_sort_keys)} label result(s) did not match the input topic order. "
            f"Examples: {bad_sort_keys[:5]}"
        )

    results = sorted(
        results,
        key=lambda r: topic_order[
            (_norm_group_value(r.get(GROUPBY_FIELD, "")), _safe_topic_id(r.get("topic_id", -1)))
        ],
    )

    with open(OUT("analysis", "llm_topic_labels.json"), "w", encoding="utf-8") as f:
        json.dump(results, f, indent=2, ensure_ascii=False, default=str)

    print(f"\n{len(results)} labels saved")

    # In legacy mode, labels_df is built in the post-processing cell as before.
    # unit_labels_df is not defined; downstream legacy-mode code should not
    # reference it.

In [ ]:
# ── Label post-processing ──────────────────────────────────────────────────
# Keep only parseable labeling results and validate them against source topics.

parse_errors = [r for r in results if r.get("parse_error")]
if parse_errors:
    print(f"WARNING: {len(parse_errors)} topics failed JSON parse — excluded from synthesis:")
    for e in parse_errors:
        print(f"  {e.get(GROUPBY_FIELD, '?')} / topic {e.get('topic_id', '?')}")

labels_df = pd.DataFrame([r for r in results if not r.get("parse_error")])

assert GROUPBY_FIELD in labels_df.columns, (
    f"labels_df missing '{GROUPBY_FIELD}' — re-run labeling for this groupby field"
)

# Source-of-truth allowed groups come from the source topic table, not model output.
ALLOWED_GROUP_VALUES = [
    str(g)
    for g in topics_df[GROUPBY_FIELD].dropna().drop_duplicates().tolist()
]

invalid_groups = sorted(
    set(labels_df[GROUPBY_FIELD].astype(str).unique()) - set(ALLOWED_GROUP_VALUES)
)
if invalid_groups:
    raise ValueError(
        f"Invalid {GROUPBY_FIELD} values returned during labeling: {invalid_groups}"
    )

# Normalize types after validation so later joins are stable.
labels_df[GROUPBY_FIELD] = labels_df[GROUPBY_FIELD].astype(str)
labels_df["topic_id"] = labels_df["topic_id"].astype(int)

labeled_groups = set(labels_df[GROUPBY_FIELD].unique()) if not labels_df.empty else set()
LABELING_FAILED_GROUPS = sorted(set(ALLOWED_GROUP_VALUES) - labeled_groups)

labels_df.groupby("coherence_flag").size().rename("count").to_frame()
labels_df[[GROUPBY_FIELD, "topic_id", "proposed_label", "coherence_flag", "description"]]

---
## Step 6 — Synthesis

Produces two synthesis passes in sequence:

1. **Cross-group synthesis** — a single LLM call over all topic lines to identify
   patterns, contrasts, framing logic, and notable boundaries across the corpus.
2. **Per-group synthesis** — one LLM call per group, run in parallel, to surface
   each group's dominant internal mechanisms.

Both passes produce plain text saved to `analysis/llm_synthesis_*.txt` and are
assembled in Step 7 as the evidence base for the structured JSON insight call.

In [ ]:
# ── SYNTHESIS — cross-group + per-group loops ─────────────────────────────
# First synthesize the whole landscape, then synthesize each group in parallel.

SYNTHESIS_SYSTEM = (
    "You are a senior program analyst at an educational nonprofit. "
    "Your job is to synthesize topic-level evidence into stable, decision-useful analytic findings "
    "for internal strategy work. You are not writing polished external copy and you are not doing creative interpretation. "
    "You are performing disciplined evidence grouping.\n\n"

    "Core objective:\n"
    "- Produce findings that are specific, well-grounded, and tightly tied to the supplied topic lines.\n"
    "- Identify useful classroom, student, operational, funding, or context patterns without forcing surprise or contrast.\n"
    "- Literal evidence discipline is more important than elegance. Do not invent recency, current-events framing, causality, or prevalence that the topic terms and labels do not support.\n\n"

    "Evidence rules:\n"
    "- Treat the supplied topic lines as the full evidence base for this step.\n"
    "- Every finding must be supported by explicitly named topics.\n"
    "- Use only topics that directly support the claim.\n"
    "- Separate direct evidence from interpretation. Direct evidence is what the topic labels, terms, and descriptions show. Interpretation is what role that evidence appears to play for students, classrooms, or school-day operations.\n"
    "- Do not generalize beyond the named supporting topics.\n"
    "- If evidence is borderline, omit the finding rather than stretching it.\n\n"

    "Substrate rules:\n"
    "- Food, hygiene, clothing, seating, storage, sensory tools, calm spaces, headphones, basic supplies, and generic devices are common classroom substrate.\n"
    "- Do not promote common substrate as a finding unless the supplied topics show a mechanism specific to this group or cross-group pattern.\n"
    "- If substrate appears only as generic classroom operating support, treat it as background rather than as a standalone finding.\n\n"

    "Merge rules:\n"
    "- Merge topics only when they share the same dominant mechanism, classroom function, student condition, or school-day operating pressure.\n"
    "- Do not merge topics just because they involve the same product category, school setting, or broad theme.\n"
    "- Keep access/accommodation, regulation/behavior, maintenance/operations, identity/belonging, safety/security, mental health context, and routine/workflow separate unless the evidence clearly shows they are the same pattern.\n"
    "- If one candidate finding is broader and another is a more specific evidence-grounded version, prefer the more specific one.\n\n"

    "Scope rules:\n"
    "- Mark sparse or sensitive-context findings as narrow or emerging.\n"
    "- Do not use broad language such as 'teachers are' when the evidence is concentrated in a small direct-signal slice.\n"
    "- Do not use trend language such as 'growing,' 'rising,' 'increasing,' or 'more often' unless the supplied evidence includes an explicit time comparison.\n\n"

    "Support-citation rules:\n"
    "- For every finding, include a 'Supporting topics' line.\n"
    "- Each supporting topic must be written exactly as <group_value>|<topic_id>.\n"
    "- group_value must be copied verbatim from the topic lines provided. Do not abbreviate, paraphrase, normalize spelling, or invent names.\n"
    "- Do not write labels or prose in place of the support references.\n\n"

    "Writing rules:\n"
    "- Be precise, concrete, and analytical.\n"
    "- Do not mention NMF, model, cluster, pipeline, or analytical machinery.\n"
    "- Do not write donor-facing rhetoric.\n"
    "- Do not force a 'looks like X, but is really Y' structure.\n"
    "- Do not manufacture a reader mistake or surface misread.\n"
    "- Do not use em dashes.\n"
    "- Do not use named places.\n"
    "- Do not invent a cleaner pattern than the evidence supports."
)

topic_lines = build_topic_lines(
    labels_df,
    GROUPBY_FIELD,
    top_terms_count=SYNTHESIS_TOP_TERMS_COUNT,
)

SYNTHESIS_PROMPT_CROSS_GROUP = f"""
Below is a list of topic lines discovered from teacher project request essays on DonorsChoose,
grouped by "{GROUPBY_FIELD}" ({GROUP_DESCRIPTION}).

Each line contains:
- group value
- topic number
- topic label
- coherence flag
- top terms
- one-sentence description

Topic lines:
{topic_lines}

Your task:
Synthesize the topic lines into a stable cross-group analysis that will later be used to generate
evidence-grounded insights. Preserve nuance, but be disciplined about grouping.

Follow this exact workflow:
1. Review the topic lines in the order given.
2. Identify candidate cross-group findings only when multiple topics share the same dominant mechanism,
   classroom function, student condition, or school-day operating pressure.
3. Keep distinct mechanisms separate even when they live in similar product categories.
4. Rank findings by:
   a. specificity of the shared mechanism
   b. clarity of distinction from nearby themes
   c. evidence strength in the topic labels, terms, and descriptions
   d. breadth across groups
5. Treat common classroom substrate as background unless the evidence shows a distinctive cross-group mechanism.
6. Omit weak or borderline patterns rather than padding.
7. A narrower but more distinctive pattern is preferred over a broader but generic one, provided both are well-supported.

Return plain text only, using exactly this structure and these section headings:

CROSS-GROUP FINDINGS
Finding 1
Title: ...
Direct evidence: ...
Interpretation: ...
Scope: ...
Supporting topics: <group_value>|<topic_id>; <group_value>|<topic_id>; <group_value>|<topic_id>

Finding 2
Title: ...
Direct evidence: ...
Interpretation: ...
Scope: ...
Supporting topics: ...

IMPORTANT DISTINCTIONS
Finding 1
Title: ...
Direct evidence: ...
Interpretation: ...
Scope: ...
Supporting topics: ...

BOUNDARIES OR NON-CENTRAL SIGNALS
Finding 1
Title: ...
Direct evidence: ...
Interpretation: ...
Scope: ...
Supporting topics: ...

Hard requirements:
- CROSS-GROUP FINDINGS: return {CROSS_MIN_INSIGHTS} to {CROSS_MAX_INSIGHTS} findings
- IMPORTANT DISTINCTIONS: return 1 to 4 findings
- BOUNDARIES OR NON-CENTRAL SIGNALS: return 0 to 3 findings
- Every finding must include a Supporting topics line
- Supporting topics must use exact <group_value>|<topic_id> format
- Do not use bullet points
- Do not skip section headings
- Do not repeat the same pattern in multiple sections
- Do not force contrast language when the evidence supports a plain pattern
- Prefer mechanism-level explanations over broad summaries like access, engagement, or support
- If a pattern is supported by only one group, it does not belong in CROSS-GROUP FINDINGS
- If a distinction is real but narrow, put it in IMPORTANT DISTINCTIONS rather than inflating it into a cross-group signature
- A boundary or non-central signal may become important if it prevents overgeneralization, distinguishes a sparse emerging topic, or clarifies what the category should not be used to claim
- If a signal is sparse, sensitive, or narrow, say so in Scope
- Do not use trend language unless explicit time-comparison evidence is supplied
- Do not use named places
- Do not use em dashes

Write like a rigorous internal analyst, not like a speechwriter.
""".strip()

PER_GROUP_INSTRUCTIONS = """
Your task:
Identify the strongest, most decision-useful patterns within this single group.

Follow this exact workflow:
1. Review the topic lines in the order given.
2. Identify the dominant subthemes in the group.
3. Separate nearby themes unless they clearly share the same dominant mechanism, classroom function, student condition, or operating pressure.
4. Prefer narrower evidence-grounded findings over broad category summaries.
5. Treat common classroom substrate as background unless the group evidence shows a distinctive mechanism.
6. Omit weak or redundant findings rather than padding.

Return plain text only, using exactly this structure and these section headings:

CORE GROUP FINDINGS
Finding 1
Title: ...
Direct evidence: ...
Interpretation: ...
Scope: ...
Supporting topics: <group_value>|<topic_id>; <group_value>|<topic_id>

Finding 2
Title: ...
Direct evidence: ...
Interpretation: ...
Scope: ...
Supporting topics: ...

IMPORTANT INTERNAL DISTINCTIONS
Finding 1
Title: ...
Direct evidence: ...
Interpretation: ...
Scope: ...
Supporting topics: ...

BOUNDARIES OR NON-CENTRAL SIGNALS
Finding 1
Title: ...
Direct evidence: ...
Interpretation: ...
Scope: ...
Supporting topics: ...

Hard requirements:
- CORE GROUP FINDINGS: return 2 to 5 findings
- IMPORTANT INTERNAL DISTINCTIONS: return 0 to 3 findings
- BOUNDARIES OR NON-CENTRAL SIGNALS: return 0 to 2 findings
- Every finding must include a Supporting topics line
- Supporting topics must use exact <group_value>|<topic_id> format
- Do not use bullet points
- Do not skip section headings
- Do not repeat the same idea across multiple sections
- Use mixed topics carefully; do not let one mixed topic dominate a whole finding unless the collision itself is the finding
- If one theme is primary and another is secondary, keep the finding centered on the primary theme
- Do not give generic summaries of the group label
- Do not force a reveal, contrast, or looks-like-X-but-is-Y structure
- Do not use trend language unless explicit time-comparison evidence is supplied
- Do not use named places
- Do not use em dashes
- If the signal is sparse, sensitive, or narrow, say so in Scope

Write like a rigorous internal analyst, not like a speechwriter.
""".strip()

synthesis_cross = _call_with_retry(
    SYNTHESIS_PROMPT_CROSS_GROUP,
    client=client,
    model_name=MODEL_SYNTHESIS,
    system_prompt=SYNTHESIS_SYSTEM,
    max_retries=MAX_RETRIES,
)
if synthesis_cross is None:
    append_warning(
        WARNINGS_PATH,
        "03_insights_generation",
        "SYNTHESIS_CROSS_GROUP_FAILED",
        "Cross-group synthesis failed",
        context={},
    )
    raise RuntimeError("Cross-group synthesis failed; stopping before downstream cells.")

with open(OUT("analysis", "llm_synthesis_cross_group.txt"), "w", encoding="utf-8") as f:
    f.write(synthesis_cross)

# Excluded groups were removed from df in the Parameters cell; they cannot
# appear in ALLOWED_GROUP_VALUES. Filter only for label coverage.
groups = [
    g for g in ALLOWED_GROUP_VALUES
    if g in set(labels_df[GROUPBY_FIELD].unique())
]

per_group_results = {}
SYNTHESIS_FAILED_GROUPS = []

with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    futures = {
        executor.submit(
            synthesize_one_group,
            g,
            labels_df=labels_df,
            groupby_field=GROUPBY_FIELD,
            group_description=GROUP_DESCRIPTION,
            per_group_instructions=PER_GROUP_INSTRUCTIONS,
            client=client,
            model_name=MODEL_SYNTHESIS,
            system_prompt=SYNTHESIS_SYSTEM,
            warnings_path=WARNINGS_PATH,
            outpath_func=OUT,
            max_retries=MAX_RETRIES,
        ): g
        for g in groups
    }
    for future in as_completed(futures):
        submitted_group = futures[future]
        try:
            group, result = future.result()
        except Exception as e:
            SYNTHESIS_FAILED_GROUPS.append(str(submitted_group))
            append_warning(
                WARNINGS_PATH,
                "03_insights_generation",
                "SYNTHESIS_GROUP_FAILED",
                f"Synthesis failed for group '{submitted_group}'",
                context={"group": submitted_group, "error": str(e)},
            )
            print(f"FAILED: {submitted_group} | error: {e}")
            continue

        if result is not None:
            per_group_results[group] = result
            print(f"Done: {group}")
        else:
            SYNTHESIS_FAILED_GROUPS.append(str(group))
            print(f"FAILED: {group}")

# Preserve the original group order before handing results downstream.
per_group_results = {g: per_group_results[g] for g in groups if g in per_group_results}

if not per_group_results:
    raise RuntimeError("No per-group synthesis results were produced; stopping before downstream cells.")


---
## Step 7 — Structured Insight Extraction with Metadata Scope Signals

This step now runs in two model passes:

1. **Draft structured extraction**: convert synthesis into structured candidate
   insight objects with source topics.
2. **Metadata-aware finalization**: compute significant metadata lift for each
   draft candidate's supporting projects, then ask the model to revise only when
   the metadata helps scope the claim or avoid overgeneralization.

Metadata lift is optional guidance. It is not causal evidence and should not
force the model to mention a segment unless doing so improves accuracy.


In [ ]:
# ── EXTERNAL-FACING INSIGHTS — structured JSON call with metadata lift ─────
# Pass 1: draft insights from synthesis.
# Pass 2: finalize those same insights with optional metadata scope signals.

OUTPUT_GROUP_KEY = "by_group"
REQUIRED_GROUP_VALUES = list(per_group_results.keys())

if not REQUIRED_GROUP_VALUES:
    raise RuntimeError("No synthesized group results are available; cannot build external-facing insights.")

synthesis_parts = []
if synthesis_cross:
    synthesis_parts.append(f"=== CROSS-GROUP ANALYSIS ===\n{synthesis_cross}")
for group_value, text in per_group_results.items():
    if text:
        synthesis_parts.append(f"=== {group_value} ===\n{text}")
synthesis_input = "\n\n".join(synthesis_parts)

INSIGHTS_SYSTEM = (
    "You are a senior program analyst at DonorsChoose. "
    "You turn structured internal analysis into clear, evidence-grounded insight objects for review by "
    "foundation leaders, corporate partners, policymakers, executives, and major donors. "
    "You return ONLY valid JSON. No preamble, no explanation, no markdown fences.\n\n"

    "Your job is not to write dramatic reveals. Your job is to state what the evidence supports, "
    "explain the classroom or student-facing mechanism, and make clear how far the claim should be generalized.\n\n"

    "Evidence discipline:\n"
    "- Use the structured synthesis as the primary evidence source.\n"
    "- Treat synthesized Direct evidence, Interpretation, Scope, and Supporting topics as evidence units.\n"
    "- Every insight must be grounded in the strongest matching supporting topics.\n"
    "- Do not infer broad support if the synthesis only gives narrow support.\n"
    "- Prefer fewer well-matched source_topics over padding with weak ones.\n"
    "- Do not merge adjacent synthesized findings unless they clearly support the same final insight.\n"
    "- Do not use project-topic support counts as proof that every project expresses the full final claim.\n\n"

    "Metadata scope-signal discipline:\n"
    "- Optional metadata scope signals may be supplied after draft extraction.\n"
    "- You are not required to mention metadata scope signals.\n"
    "- Metadata facts were selected using effect-size thresholds; the prompt reports shares, percentage-point differences, and lift only for readability.\n"
    "- Do not treat metadata lift as causal evidence.\n"
    "- Do not create a new insight from metadata lift alone.\n"
    "- Do not mention a grade band, geography, project category, school context, or project-size segment unless it is supported by the source topics, direct evidence, or metadata scope signals.\n"
    "- If supporting projects strongly over-index in one segment, avoid language implying the pattern is evenly distributed across all projects.\n\n"

    "When and how to use metadata scope signals:\n"
    "- If the optional metadata scope signals materially narrow where the candidate insight is concentrated, reflect the strongest relevant signal in scope_or_caveat.\n"
    "- Treat a signal as materially narrowing when the supporting projects are at least roughly twice as concentrated in one segment as in the run as a whole (the 'X.Yx' multiplier shown in each fact).\n"
    "- For binary or yes/no dimensions, apply a second rule: treat a signal as materially narrowing when the supporting projects' share differs from the run baseline by at least 15 percentage points, even if the multiplier is below 2x.\n"
    "- Do not mention every metadata fact.\n"
    "- Do not reflect a metadata signal that simply restates the topic of the insight; for example, a books or reading insight concentrating in a literacy-related category, or a flexible-seating insight concentrating in a classroom-basics category, is tautological, not scope-narrowing.\n"
    "- If the metadata does not materially change the scope, leave the insight scoped by the source topics only.\n"
    "- Metadata should usually appear only in scope_or_caveat; title, finding, evidence_basis, and why_it_matters should remain based on topic evidence unless the same scope constraint is already explicit in the topic terms or labels.\n\n"

    "Insight selection:\n"
    "- A valid insight may be a plain mechanism, a meaningful split, a boundary, a sparse emerging signal, or a practical funding/reporting implication.\n"
    "- Do not force every insight into a looks-like-X-but-is-Y structure.\n"
    "- Do not manufacture a reader mistake or surface-level misread.\n"
    "- Avoid observations that merely restate what the group label already implies.\n"
    "- Prefer the smallest non-overlapping set of insights that captures the strongest supported patterns.\n"
    "- Never add an insight just because there is room before the maximum count.\n"
    "- If two possible insights have the same practical implication, merge them or keep the stronger one.\n\n"

    "Substrate rule:\n"
    "- Food, hygiene, clothing, seating, storage, sensory tools, calm spaces, headphones, basic supplies, and generic devices are common classroom substrate.\n"
    "- Do not promote common substrate as a main insight unless the evidence shows a mechanism specific to the current group or cross-group pattern.\n"
    "- If substrate appears only as generic classroom operating support, mention it only as evidence, background, or scope.\n\n"

    "DonorsChoose voice:\n"
    "- Phrase findings around students, classrooms, or classroom conditions when that is natural and evidence-faithful.\n"
    "- Use teachers as the subject when the evidence is genuinely about what teachers request, build, adapt, manage, or describe.\n"
    "- Do not write awkward agentless prose just to avoid teacher-first phrasing.\n"
    "- The implication should connect back to students or classroom conditions when possible, but the sentence subject should be whatever makes the claim clearest.\n"
    "- Write directly, warmly, and concretely. Warmth should come from classroom materials, routines, and student conditions, not emotional adjectives.\n"
    "- Do not use named places.\n"
    "- Do not use em dashes.\n"
    "- Do not use jargon or corporate boilerplate.\n"
    "- Do not use phrases such as 'not just X,' 'at its core,' 'this speaks to,' 'it is clear that,' 'leverage,' 'utilize,' 'facilitate,' or 'impactful' as a standalone adjective.\n\n"

    "Scope rules:\n"
    "- Use cautious scope language for sparse, sensitive, or narrow groups.\n"
    "- Use phrases such as 'in this slice,' 'among these projects,' or 'a small but coherent signal' when support is limited.\n"
    "- Do not use trend language such as 'growing,' 'rising,' 'increasing,' or 'more often' unless the supplied evidence includes an explicit time comparison.\n\n"

    "Output rules:\n"
    "- Return ONLY valid JSON.\n"
    "- Use exactly the required schema and fields.\n"
    "- Do not add extra fields.\n"
    "- Do not include markdown fences, commentary, or explanation outside the JSON object."
)

example_source_topics = []
example_rows = (
    labels_df[[GROUPBY_FIELD, "topic_id"]]
    .drop_duplicates()
    .head(3)
    .to_dict("records")
)
for row in example_rows:
    example_source_topics.append(f"{row[GROUPBY_FIELD]}|{int(row['topic_id'])}")
if not example_source_topics:
    for i, gv in enumerate(REQUIRED_GROUP_VALUES[:2], start=1):
        example_source_topics.append(f"{gv}|{i}")
example_source_topics_json = json.dumps(example_source_topics, ensure_ascii=False)

INSIGHT_SCHEMA_INSTRUCTIONS = f"""
INSIGHT STRUCTURE:
For every insight, use exactly these fields:

- title:
  A short, plain mechanism statement. Usually center students, classrooms,
  classroom conditions, teacher actions, or school-day operations, whichever is
  clearest and most evidence-faithful. Do not write a slogan. Do not force a
  contrast frame. Do not use "really about," "not just," "more than," "hidden,"
  or "easy to miss."

- finding:
  1-3 sentences explaining the pattern in direct, concrete language. State what
  the evidence supports. Phrase findings around students, classrooms, or
  classroom conditions when natural and evidence-faithful. Use teachers as the
  subject when the evidence is genuinely about what teachers request, build,
  adapt, manage, or describe. Match length to the complexity of the signal. Do
  not pad simple findings.

- evidence_basis:
  1-2 sentences naming the concrete evidence behind the finding: materials,
  classroom routines, project contexts, repeated terms, student conditions, or
  topic descriptions. This should make clear what the interpretation rests on.
  Do not simply repeat the category name.

- scope_or_caveat:
  1 sentence explaining where the claim is strongest, narrowest, or most
  limited. This is the preferred place to use optional metadata scope signals
  when they improve accuracy. If there is no major caveat, state the specific
  group, mechanism, or evidence base where the pattern is strongest. Do not
  write generic filler such as "This pattern may not apply everywhere."

- why_it_matters:
  1 sentence explaining the practical implication for interpretation, funding,
  reporting, partnership strategy, or policy understanding. This field is not a
  call to action and should not be donor-marketing copy.

- source_topics:
  A list of the strongest grounding topics for the insight. Use only topics
  that directly support the claim. Usually 2-4 topics is enough, but do not force
  a fixed count. Prefer fewer strong matches over weak padding. Required format:
    - each item must be a string
    - each string must be exactly "<group_value>|<topic_id>"
    - group_value must exactly match one of the group values shown in the synthesis
    - topic_id must be the integer topic number for that group
  Example for this run: {example_source_topics_json}

SOURCE_TOPIC RULES:
- Prefer source_topics that are explicitly named in Supporting topics lines.
- If a synthesized finding names multiple supporting topics but only some
  directly support your final claim, include only the strongest matches.
- If a topic is illustrative but not necessary to support the claim, leave it out.
- Do not use labels, prose, section names, or invented group names in place of group_value.
- source_topics must be a list of strings only, not objects.
- Do not include any extra fields beyond: title, finding, evidence_basis,
  scope_or_caveat, why_it_matters, source_topics.

STYLE:
- Direct, warm, concrete, and human.
- Not academic.
- Not corporate boilerplate.
- Not donor-marketing fluff.
- No named places.
- No em dashes.
- No mention of topics, models, clusters, prompts, or analytical machinery in
  title, finding, evidence_basis, scope_or_caveat, or why_it_matters.
- Do not manufacture a "why this is easy to miss" explanation.
- Do not use "may suggest" or "could indicate," but do use scope language when support is narrow.
- No filler.
""".strip()

DRAFT_PROMPT = f"""
You are given a structured synthesis of classroom project request patterns.

The synthesis is grouped by the field "{GROUPBY_FIELD}" ({GROUP_DESCRIPTION}).

It contains:
- cross-group findings
- within-group findings
- important distinctions
- boundaries or non-central signals
- explicit Supporting topics lines in the format <group_value>|<topic_id>

Structured synthesis:
{synthesis_input}

Your task:
Produce a draft evidence-grounded insights document for review by external-facing
DonorsChoose audiences: foundation leaders, corporate partners, policymakers,
executives, and major donors.

The best insights should help a reader understand what students and classrooms
need, what school-day conditions are shaping those needs, and what funding or
reporting implications follow. Do not force surprise. A plain, well-scoped
finding is better than a dramatic reveal.

WORKING METHOD:
1. Read the structured synthesis as a set of evidence units, not just as prose.
2. Identify plausible candidate cross-group insights and group-specific insights.
3. For each candidate, separate direct evidence, interpretation, and scope.
4. Remove or merge any candidate whose core implication materially overlaps another candidate.
5. Keep only the strongest non-overlapping set.
6. Then generate the JSON.

SELECTION CRITERIA:
- Prioritize insights with a clear classroom or student-facing mechanism.
- Prioritize insights that matter for funding strategy, partnership design, policy
  interpretation, reporting, or external messaging.
- Current K-12 dynamics such as AI use in classrooms, post-pandemic recovery,
  technology continuity, cellphone management, transition planning, ESSER-cliff
  budget pressure, staffing strain, attendance, safety, and mental health are
  valuable only when directly supported by the synthesis.
- Avoid observations that merely restate what the group label already implies.
- Avoid methodological commentary and data-practitioner-only insights.
- Do not promote common classroom substrate as a main insight unless the evidence
  shows a mechanism specific to this group or pattern.
- Do not use trend language unless explicit time-comparison evidence is supplied.
- A boundary or non-central signal may become an insight if it prevents
  overgeneralization, distinguishes a sparse emerging topic, or clarifies what
  the category should not be used to claim.

DISTINCTNESS RULE:
- Prefer the smallest non-overlapping set of insights that captures the strongest supported patterns.
- Do not add an insight just because room remains before the maximum count.
- Each additional insight must materially change what the reader learns.
- Avoid returning multiple phrasings of the same mechanism, material need, or scope caveat.

MERGE RULE:
- If two candidate insights rely on substantially overlapping source topics and lead to the same practical implication, merge them or keep only the stronger one.
- If a group-specific insight is just a narrower restatement of a cross-group insight, keep it only when the narrower version adds concrete detail.
- Prefer fewer, sharper insights over exhaustive coverage of adjacent themes.

COUNT + COVERAGE REQUIREMENTS:
- Return between {CROSS_MIN_INSIGHTS} and {CROSS_MAX_INSIGHTS} cross-group insights in "key_insights".
- Treat {CROSS_MIN_INSIGHTS} as the default target.
- Add more than {CROSS_MIN_INSIGHTS} only if an additional insight is clearly
  distinct, non-overlapping, strongly supported, and would materially change what
  the reader learns.

- "{OUTPUT_GROUP_KEY}" must include every group value in this exact list:
  {json.dumps(REQUIRED_GROUP_VALUES, ensure_ascii=False)}

- Return between {PER_GROUP_MIN_INSIGHTS} and {PER_GROUP_MAX_INSIGHTS} strongest defensible insights for each group value above.
- Treat {PER_GROUP_MIN_INSIGHTS} as the default target for each group.
- Do not omit group values silently.
- Prefer slightly weaker but still defensible coverage over omission.
- Keep by-group insights focused on the group's dominant internal mechanisms and
  avoid repeating the same implication in multiple phrasings.

{INSIGHT_SCHEMA_INSTRUCTIONS}

VALIDITY RULES:
- Return valid JSON only.
- The response is incomplete if any required group value is missing from "{OUTPUT_GROUP_KEY}".
- Do not return markdown fences or explanatory text.
- Every source_topics item must follow the exact required format.
- Every insight object must contain exactly these six fields:
  title, finding, evidence_basis, scope_or_caveat, why_it_matters, source_topics

Return a JSON object with exactly this structure:
{{
  "key_insights": [/* {CROSS_MIN_INSIGHTS}-{CROSS_MAX_INSIGHTS} insight objects */],
  "{OUTPUT_GROUP_KEY}": {{
    "<group value>": [/* {PER_GROUP_MIN_INSIGHTS}-{PER_GROUP_MAX_INSIGHTS} insight objects */],
    ...
  }}
}}
""".strip()

def _normalize_insights_response(raw_obj, output_group_key, required_group_values):
    if not isinstance(raw_obj, dict):
        raise ValueError("Top-level insights response is not a JSON object.")

    raw_key = raw_obj.get("key_insights", [])
    raw_by_group = raw_obj.get(output_group_key, {})

    if not isinstance(raw_key, list):
        raise ValueError("key_insights is not a list.")
    if not isinstance(raw_by_group, dict):
        raise ValueError(f"{output_group_key} is not an object.")

    normalized = {
        "key_insights": [
            normalize_insight(i, required_group_values=required_group_values)
            for i in raw_key
        ],
        output_group_key: {},
    }

    missing_group_values = [g for g in required_group_values if g not in raw_by_group]
    if missing_group_values:
        raise ValueError(f"Model omitted required group values: {missing_group_values}")

    extra_group_values = [g for g in raw_by_group.keys() if g not in required_group_values]
    if extra_group_values:
        print(f"WARNING: extra group values returned and ignored: {extra_group_values}")

    for group_value in required_group_values:
        items = raw_by_group.get(group_value, [])
        if not isinstance(items, list):
            raise ValueError(f"{output_group_key}['{group_value}'] is not a list.")
        normalized[output_group_key][group_value] = [
            normalize_insight(i, required_group_values=required_group_values)
            for i in items
        ]
    return normalized

def _validate_insight_counts(data, output_group_key):
    if not (CROSS_MIN_INSIGHTS <= len(data["key_insights"]) <= CROSS_MAX_INSIGHTS):
        raise ValueError(
            f"Expected between {CROSS_MIN_INSIGHTS} and {CROSS_MAX_INSIGHTS} key insights, "
            f"got {len(data['key_insights'])}"
        )
    bad_group_counts = {
        group_value: len(items)
        for group_value, items in data[output_group_key].items()
        if not (PER_GROUP_MIN_INSIGHTS <= len(items) <= PER_GROUP_MAX_INSIGHTS)
    }
    if bad_group_counts:
        raise ValueError(
            f"Expected between {PER_GROUP_MIN_INSIGHTS} and {PER_GROUP_MAX_INSIGHTS} insights per group value, "
            f"got: {bad_group_counts}"
        )

def _iter_all_insights(data, output_group_key):
    for idx, item in enumerate(data.get("key_insights", []), start=1):
        yield f"KI_{idx:03d}", "key_insights", None, item
    for group_value, items in data.get(output_group_key, {}).items():
        for idx, item in enumerate(items, start=1):
            yield f"BG_{group_value}_{idx:03d}", output_group_key, group_value, item

# Pass 1: draft structured insights.
draft_resp = client.chat.completions.create(
    model=MODEL_SYNTHESIS,
    messages=[
        {"role": "system", "content": INSIGHTS_SYSTEM},
        {"role": "user", "content": DRAFT_PROMPT},
    ],
    response_format={"type": "json_object"},
)

draft_raw_json = strip_json_fences(draft_resp.choices[0].message.content)
draft_insights_raw = json.loads(draft_raw_json)
draft_insights_data = _normalize_insights_response(
    draft_insights_raw,
    OUTPUT_GROUP_KEY,
    REQUIRED_GROUP_VALUES,
)
_validate_insight_counts(draft_insights_data, OUTPUT_GROUP_KEY)

write_json(
    OUT("insights", "insights_candidates_draft_pre_metadata.json"),
    {
        "key_insights": [
            project_insight_for_saved_candidates(i)
            for i in draft_insights_data["key_insights"]
        ],
        OUTPUT_GROUP_KEY: {
            group_value: [
                project_insight_for_saved_candidates(i)
                for i in items
            ]
            for group_value, items in draft_insights_data[OUTPUT_GROUP_KEY].items()
        },
    },
)

# Compute candidate-level metadata scope signals from draft source topics.
BRIDGE_LOOKUP_FOR_LIFT = build_bridge_lookup(project_topic_bridge_df)

METADATA_LIFT_CONTEXT = build_metadata_lift_context(
    run_df=df,
    dimensions=METADATA_LIFT_DIMENSIONS,
    project_id_col="project_id",
) if METADATA_LIFT_ENABLED else None

candidate_packages = []
metadata_lift_rows = []

for candidate_id, section, group_value, item in _iter_all_insights(draft_insights_data, OUTPUT_GROUP_KEY):
    support_ids = candidate_support_project_ids_from_source_topics(
        item.get("source_topics", []),
        groupby_field=GROUPBY_FIELD,
        bridge_lookup=BRIDGE_LOOKUP_FOR_LIFT,
    )

    if METADATA_LIFT_ENABLED and METADATA_LIFT_CONTEXT is not None:
        lift_df = compute_metadata_lift(
            support_project_ids=support_ids,
            context=METADATA_LIFT_CONTEXT,
            thresholds=METADATA_LIFT_THRESHOLDS,
        )
    else:
        lift_df = pd.DataFrame()

    if not lift_df.empty:
        lift_df = lift_df.copy()
        lift_df.insert(0, "candidate_id", candidate_id)
        lift_df.insert(1, "section", section)
        lift_df.insert(2, "group_value", group_value)
        metadata_lift_rows.append(lift_df)

    metadata_lift_block = format_metadata_lift_facts(lift_df)
    candidate_packages.append({
        "candidate_id": candidate_id,
        "section": section,
        "group_value": group_value,
        "draft_insight": project_insight_for_saved_candidates(item),
        "supporting_project_count_for_lift": len(support_ids),
        "metadata_scope_signals": metadata_lift_block,
    })

metadata_lift_audit_df = (
    pd.concat(metadata_lift_rows, ignore_index=True)
    if metadata_lift_rows else
    pd.DataFrame(columns=[
        "candidate_id", "section", "group_value", "dimension", "value",
        "insight_project_count", "run_project_count", "insight_total_projects",
        "run_total_projects", "insight_share", "run_share", "difference_pp",
        "abs_difference_pp", "lift", "cohens_h",
    ])
)
metadata_lift_audit_df.to_csv(
    OUT("insights", "metadata_lift_facts_for_step7.csv"),
    index=False,
)

print(
    f"Metadata scope signals prepared for "
    f"{sum(bool(p.get('metadata_scope_signals')) for p in candidate_packages):,} "
    f"of {len(candidate_packages):,} draft candidates."
)

# Pass 2: finalize insight language with optional metadata scope signals.
HAS_METADATA_SCOPE_SIGNALS = any(
    bool(p.get("metadata_scope_signals"))
    for p in candidate_packages
)

if HAS_METADATA_SCOPE_SIGNALS:
    FINALIZE_PROMPT = f"""
You are finalizing draft insight objects using optional metadata scope signals.

This is a wording and scope pass only.
Do not generate new insights.
Do not remove insights.
Do not split or merge insights.
Do not change the number, order, sections, or group buckets.
Do not add, remove, reorder, or edit source_topics.
Do not edit source_topics_claimed.

Use metadata only when it helps prevent overbroad wording.
Metadata is not causal evidence.
Metadata should usually affect scope_or_caveat.
Only revise finding, evidence_basis, or why_it_matters if the draft clearly overstates scope.
If metadata does not materially improve an insight, preserve the draft wording.

Run context:
- Grouping field: {GROUPBY_FIELD}
- Group description: {GROUP_DESCRIPTION}
- Strategic loop enabled: {STRATEGIC_LOOP_ENABLED}
- Strategic run metadata: {json.dumps({k: STRATEGIC_RUN_META.get(k) for k in ['strategic_area_id', 'strategic_area_label', 'split_id', 'groupby_fields', 'is_strategic_injected_tag']}, ensure_ascii=False)}

Draft candidates with optional metadata scope signals:
{json.dumps(candidate_packages, ensure_ascii=False, indent=2)}

Return valid JSON only.
Return the same top-level structure as the drafts:
{{
  "key_insights": [/* same number and order as draft key_insights */],
  "{OUTPUT_GROUP_KEY}": {{
    "<group value>": [/* same number and order as draft group insights */],
    ...
  }}
}}

Each returned insight must keep exactly these fields:
title, finding, evidence_basis, scope_or_caveat, why_it_matters, source_topics

source_topics must be copied exactly from the draft for each insight.
source_topics_claimed, if present in the draft package, must not be changed.
""".strip()

    final_resp = client.chat.completions.create(
        model=MODEL_SYNTHESIS,
        messages=[
            {"role": "system", "content": INSIGHTS_SYSTEM},
            {"role": "user", "content": FINALIZE_PROMPT},
        ],
        response_format={"type": "json_object"},
    )

    final_raw_json = strip_json_fences(final_resp.choices[0].message.content)
    insights_data = _normalize_insights_response(
        json.loads(final_raw_json),
        OUTPUT_GROUP_KEY,
        REQUIRED_GROUP_VALUES,
    )
    _validate_insight_counts(insights_data, OUTPUT_GROUP_KEY)
else:
    print("No metadata lift facts passed Step 7 thresholds; skipping metadata-scope refinement pass.")
    insights_data = draft_insights_data

HARD_STYLE_PATTERNS = [
    r"—",
    r"\bnot just\b",
    r"\bat its core\b",
    r"\bthis speaks to\b",
    r"\bit is clear that\b",
    r"\bperhaps most importantly\b",
    r"\bin a world where\b",
]

SOFT_STYLE_PATTERNS = [
    r"\bleverage\b",
    r"\butilize\b",
    r"\bfacilitate\b",
    r"\bimpactful\b",
    r"\bstakeholders\b",
    r"\bend users\b",
    r"\bbeneficiaries\b",
    r"\ba significant number of\b",
]

def _pattern_hits(text, patterns):
    return [p for p in patterns if re.search(p, text, flags=re.IGNORECASE)]

for _, section, group_value, item in _iter_all_insights(insights_data, OUTPUT_GROUP_KEY):
    combined = " ".join(
        str(item.get(k, "") or "")
        for k in ["title", "finding", "evidence_basis", "scope_or_caveat", "why_it_matters"]
    )

    hard_hits = _pattern_hits(combined, HARD_STYLE_PATTERNS)
    soft_hits = _pattern_hits(combined, SOFT_STYLE_PATTERNS)

    if hard_hits:
        append_warning(
            WARNINGS_PATH,
            "03_insights_generation",
            "INSIGHT_HARD_STYLE_VIOLATION",
            "Insight contains hard-banned DonorsChoose style pattern",
            context={
                "section": section,
                "group_value": group_value,
                "title": item.get("title", ""),
                "violations": hard_hits,
            },
        )
        print(f"HARD STYLE WARNING: {item.get('title', '')[:80]} | {hard_hits}")

    if soft_hits:
        append_warning(
            WARNINGS_PATH,
            "03_insights_generation",
            "INSIGHT_SOFT_STYLE_WARNING",
            "Insight contains soft-flagged DonorsChoose style pattern",
            context={
                "section": section,
                "group_value": group_value,
                "title": item.get("title", ""),
                "violations": soft_hits,
            },
        )
        print(f"SOFT STYLE WARNING: {item.get('title', '')[:80]} | {soft_hits}")

insights_candidates_for_save = {
    "key_insights": [
        project_insight_for_saved_candidates(i)
        for i in insights_data["key_insights"]
    ],
    OUTPUT_GROUP_KEY: {
        group_value: [
            project_insight_for_saved_candidates(i)
            for i in items
        ]
        for group_value, items in insights_data[OUTPUT_GROUP_KEY].items()
    },
}

# Mid-cell write required by spec: post-normalize_insight(), pre-verification.
write_json(OUT("insights", "insights_candidates.json"), insights_candidates_for_save)

print(
    f"Structured insight extraction complete: "
    f"{len(insights_data['key_insights'])} key insights, "
    f"{sum(len(v) for v in insights_data[OUTPUT_GROUP_KEY].values())} by-group insights, "
    f"{len(metadata_lift_audit_df)} metadata lift facts passed Step 7 thresholds."
)


---
## Step 8 — Topic Verification

Verifies that each claimed source topic genuinely supports its insight. Topics
that are too broad, indirect, or adjacent are dropped.

Verification strategy:
- **Cross-group insights** — always verified.
- **By-group insights** — verified only when they cite ≥ `by_group_min_source_topics`
  source topics; narrow local findings with 1–2 cited topics pass through without
  an additional API call.

In [ ]:
# ── Topic verification ─────────────────────────────────────────────────────
# Verify source-topic grounding for final insights.
# Strategy:
# - always verify key_insights
# - verify by-group insights only when they cite 3+ source topics
#   (broad claims are more likely to overclaim support than narrow local ones)

VERIFY_SYSTEM = (
    "You are validating whether cited topics directly support an insight. "
    "Be strict. Keep a topic only if its label and description directly support "
    "the insight's title, finding, and evidence_basis. "
    "Ignore any practical implication or downstream strategy that is not part of the finding. "
    "If support is broad, indirect, adjacent, generic substrate, or only loosely related, drop it. "
    "Prefer false negatives to false positives. "
    "Return only valid JSON."
)

VERIFY_BY_GROUP_MIN_SOURCE_TOPICS = VERIFY_CFG["by_group_min_source_topics"]

# Always verify cross-group/key insights
insights_data["key_insights"], key_verify_stats = _verify_insight_list(
    insights_data.get("key_insights", []),
    labels_df=labels_df,
    groupby_field=GROUPBY_FIELD,
    required_group_values=REQUIRED_GROUP_VALUES,
    client=client,
    model_verify=MODEL_VERIFY,
    system_prompt=VERIFY_SYSTEM,
    warnings_path=WARNINGS_PATH,
    min_source_topics_to_verify=1,
)

group_verify_stats = {}
for cat in insights_data.get(OUTPUT_GROUP_KEY, {}):
    verified_items, stats = _verify_insight_list(
        insights_data[OUTPUT_GROUP_KEY][cat],
        labels_df=labels_df,
        groupby_field=GROUPBY_FIELD,
        required_group_values=REQUIRED_GROUP_VALUES,
        client=client,
        model_verify=MODEL_VERIFY,
        system_prompt=VERIFY_SYSTEM,
        warnings_path=WARNINGS_PATH,
        min_source_topics_to_verify=VERIFY_BY_GROUP_MIN_SOURCE_TOPICS,
    )
    insights_data[OUTPUT_GROUP_KEY][cat] = verified_items
    group_verify_stats[cat] = stats

total_group_changed = sum(v["changed_count"] for v in group_verify_stats.values())
total_group_zeroed = sum(v["dropped_to_zero_count"] for v in group_verify_stats.values())
total_group_topics_before = sum(v["topics_before"] for v in group_verify_stats.values())
total_group_topics_after = sum(v["topics_after"] for v in group_verify_stats.values())

print("Verification complete.")
print(
    f"Key insights: {key_verify_stats['insight_count']} insights | "
    f"{key_verify_stats['changed_count']} changed | "
    f"{key_verify_stats['dropped_to_zero_count']} dropped to zero topics | "
    f"{key_verify_stats['topics_before']} -> {key_verify_stats['topics_after']} source topics"
)
print(
    f"By-group insights: {sum(v['insight_count'] for v in group_verify_stats.values())} insights | "
    f"{total_group_changed} changed | "
    f"{total_group_zeroed} dropped to zero topics | "
    f"{total_group_topics_before} -> {total_group_topics_after} source topics | "
    f"verified only when source_topics >= {VERIFY_BY_GROUP_MIN_SOURCE_TOPICS}"
)

---
## Step 9 — Evidence Tables

Expands each verified insight into a flat summary row (`insights_flat.csv`) and
a per-topic support detail row (`insight_topic_support.csv`). Projects are ranked
by combined topic_share across all of an insight's verified topics so Looker
links surface the most representative essays.

In [ ]:
# ── VERIFIED INSIGHT SUPPORT TABLES ────────────────────────────────────────

assert "project_topic_bridge_df" in dir() and not project_topic_bridge_df.empty, (
    "project_topic_bridge_df missing. Ensure the bridge-build cell ran successfully."
)

BRIDGE_LOOKUP = build_bridge_lookup(project_topic_bridge_df)

label_index = build_label_index(
    labels_df,
    groupby_field=GROUPBY_FIELD,
    warnings_path=WARNINGS_PATH,
)

insights_flat_df, insight_topic_support_df = build_verified_insight_tables(
    insights_data,
    OUTPUT_GROUP_KEY,
    groupby_field=GROUPBY_FIELD,
    bridge_lookup=BRIDGE_LOOKUP,
    label_index=label_index,
    run_id=RUN_ID,
    top_project_id_limit=CSV_MAX_IDS_PER_INSIGHT,
)

# Attach strategic-run metadata for downstream multi-run review/HTML work.
for frame in [insights_flat_df, insight_topic_support_df]:
    frame["strategic_loop_enabled"] = STRATEGIC_LOOP_ENABLED
    frame["strategic_area_id"] = STRATEGIC_RUN_META.get("strategic_area_id")
    frame["strategic_area_label"] = STRATEGIC_RUN_META.get("strategic_area_label")
    frame["split_id"] = STRATEGIC_RUN_META.get("split_id")
    frame["resolved_groupby_field"] = GROUPBY_FIELD

insights_flat_df.to_csv(OUT("insights", "insights_flat.csv"), index=False)
insight_topic_support_df.to_csv(
    OUT("insights", "insight_topic_support_candidates.csv"),
    index=False,
)

print(f"Candidate insights summarized: {len(insights_flat_df):,}")


---
## Step 10 — Packaging, Dedupe & Topline Selection

Three sequential operations:

1. **Packaging** — apply quality threshold gates. Insights that don't clear
   `min_verified_topic_count`, `min_supporting_project_count`, and
   `min_mean_topic_share` are rejected.
2. **Deduplication** — deterministic overlap-based dedup within pair types.
3. **Topline selection** — assigns `report_section` to each accepted insight:
   `main_cross` (top N by quality rank), `main_by_group` (top 1 per group),
   `appendix_cross`, `appendix_by_group`.

In [ ]:
# ── PACKAGING + DEDUPE + SIMPLE TOPLINE CUT ────────────────────────────────
# Business rule:
# - accept insights that clear threshold gates
# - remove obvious duplicates
# - then choose topline from the remaining pool
#
# Main section:
# - top N cross-category insights by:
#     1) supporting_project_count
#     2) verified_topic_count
#     3) mean_topic_share_all_verified_topics
# - top 1 by-group insight per category using the same ranking
#
# Appendix:
# - all other accepted insights

packaging = apply_deterministic_packaging(
    insights_flat_df,
    output_group_key=OUTPUT_GROUP_KEY,
    packaging_cfg=PACKAGING_CFG,
)

accepted_pack_df = packaging["accepted_df"]

print(f"Total insights before packaging: {len(insights_flat_df)}")
print(f"Accepted after thresholds: {len(accepted_pack_df)}")

curated_df, dedup_audit_df = dedupe_packaged_insights(
    accepted_pack_df,
    dedupe_cfg=DEDUPE_CFG,
)

curated_df = curated_df.copy()
curated_df["looker_url"] = curated_df["top_project_ids"].apply(
    lambda ids: build_looker_project_url(
        base_url=LOOKER_BASE_URL,
        project_ids=ids,
        filter_field=LOOKER_FILTER_FIELD,
        fields=LOOKER_FIELDS,
        limit=LOOKER_LIMIT,
        max_ids=LOOKER_ID_LIMIT,
    )
)

# main_min_verification_ratio gates eligibility for main-section placement;
# sourced from params.yaml → analysis.packaging.main_min_verification_ratio.
curated_df = assign_topline_sections_simple(
    curated_df,
    output_group_key=OUTPUT_GROUP_KEY,
    main_cross_limit=PACKAGING_CFG["main_cross_limit"],
    main_min_verification_ratio=MAIN_MIN_VERIFICATION_RATIO,
)

main_cross_df = curated_df[curated_df["report_section"] == "main_cross"].copy()
main_by_group_df = curated_df[curated_df["report_section"] == "main_by_group"].copy()
appendix_df = curated_df[
    curated_df["report_section"].isin(["appendix_cross", "appendix_by_group"])
].copy()

accepted_ids = set(curated_df["insight_id"])
rejected_ids = set(insights_flat_df["insight_id"]) - accepted_ids

# export project_id list by insight
insight_project_df = (
    insight_topic_support_df[
        insight_topic_support_df["insight_id"].isin(accepted_ids)
    ][["insight_id", "group_value", "topic_id"]]
    .merge(
        project_topic_bridge_df[["project_id", GROUPBY_FIELD, "topic_id"]],
        left_on=["group_value", "topic_id"],
        right_on=[GROUPBY_FIELD, "topic_id"],
        how="left",
    )
    [["insight_id", "project_id"]]
    .drop_duplicates()
    .reset_index(drop=True)
)
insight_project_df.to_csv(OUT("insights", "insight_project_bridge.csv"), index=False)

# ── EXPORT: insight-with-text + sampled essays ────────────────────────────
# Companion to insight_project_bridge.csv, but with:
#   1) insight title / finding / evidence fields instead of insight_id only
#   2) essay text instead of project_id only
#   3) capped at N projects per insight (random sample, fixed seed)
from utils import load_essay_snippet_lookup

ESSAY_SAMPLE_PER_INSIGHT = 20
ESSAY_MAX_CHARS = 1200  # bump if you want fuller essays per row

# 1) random-sample up to N projects per insight (deterministic via seed)
sampled_pairs_df = (
    insight_project_df
    .groupby("insight_id", group_keys=False)
    .apply(
        lambda g: g.sample(
            n=min(ESSAY_SAMPLE_PER_INSIGHT, len(g)),
            random_state=42,
        )
    )
    .reset_index(drop=True)
)

# 2) look up essay text for just the sampled project IDs
import re

ACTUAL_TEXT_COL = "tokens"
needed_ids = set(sampled_pairs_df["project_id"].unique())
essay_lookup: dict = {}

for fpath in sorted((ROOT / "DATA").glob("project_essays*.csv")):
    if len(essay_lookup) == len(needed_ids):
        break
    for chunk in pd.read_csv(
        fpath, usecols=["project_id", ACTUAL_TEXT_COL], chunksize=200_000
    ):
        sub = chunk[chunk["project_id"].isin(needed_ids - set(essay_lookup))]
        for _, row in sub.iterrows():
            text = re.sub(r"\s+", " ", str(row.get(ACTUAL_TEXT_COL, "") or "")).strip()
            if text:
                essay_lookup[row["project_id"]] = text[:ESSAY_MAX_CHARS]
        if len(essay_lookup) == len(needed_ids):
            break
sampled_pairs_df["essay_text"] = (
    sampled_pairs_df["project_id"].map(essay_lookup).fillna("")
)

# 3) attach the polished insight text from curated_df
insight_text_df = curated_df[[
    "insight_id", "title", "finding", "evidence_basis",
    "scope_or_caveat", "why_it_matters",
    "category_bucket", "report_section",
]].drop_duplicates(subset=["insight_id"])

insight_project_text_df = (
    sampled_pairs_df
    .merge(insight_text_df, on="insight_id", how="left")
    [[
        "insight_id", "report_section", "category_bucket",
        "title", "finding", "evidence_basis",
        "scope_or_caveat", "why_it_matters",
        "project_id", "essay_text",
    ]]
    .sort_values(["report_section", "insight_id", "project_id"])
    .reset_index(drop=True)
)

insight_project_text_df.to_csv(
    OUT("insights", "insight_project_text_sample.csv"), index=False
)
print(
    f"Wrote insight_project_text_sample.csv: "
    f"{len(insight_project_text_df):,} rows across "
    f"{insight_project_text_df['insight_id'].nunique()} insights "
    f"(up to {ESSAY_SAMPLE_PER_INSIGHT} essays each)"
)

curated_df.to_csv(OUT('chart_data', 'curated_df.csv'), index=False)

print(f"Accepted before dedupe: {len(accepted_pack_df)}")
print(f"Dropped as obvious duplicates: {len(dedup_audit_df)}")
print(f"Accepted after dedupe: {len(curated_df)}")
print(f"Main cross-category selected: {len(main_cross_df)}")
print(f"Main by-group selected: {len(main_by_group_df)}")
print(f"Appendix selected: {len(appendix_df)}")
print(f"{len(insight_project_df):,} insight-project pairs across {insight_project_df['insight_id'].nunique()} insights")

---
## Step 11 — Final Outputs & Manifests

Persists all accepted and rejected outputs, builds the DOCX report, and writes
stage and pipeline manifests. Running this cell again after a partial run is
safe — all writes are deterministic.

In [ ]:
# ── FINAL OUTPUTS + REPORTING ──────────────────────────────────────────────
# Persist accepted/rejected outputs, evidence exports, chart data, DOCX, and manifests.

structured = build_structured_from_curated(
    curated_df,
    output_group_key=OUTPUT_GROUP_KEY,
)

# Add run metadata to structured insights for future multi-run HTML/review use.
def _add_run_meta_to_structured_item(item):
    item["strategic_loop_enabled"] = STRATEGIC_LOOP_ENABLED
    item["strategic_area_id"] = STRATEGIC_RUN_META.get("strategic_area_id")
    item["strategic_area_label"] = STRATEGIC_RUN_META.get("strategic_area_label")
    item["split_id"] = STRATEGIC_RUN_META.get("split_id")
    item["resolved_groupby_field"] = GROUPBY_FIELD
    return item

structured["key_insights"] = [
    _add_run_meta_to_structured_item(i)
    for i in structured.get("key_insights", [])
]
structured[OUTPUT_GROUP_KEY] = {
    group_value: [
        _add_run_meta_to_structured_item(i)
        for i in items
    ]
    for group_value, items in structured.get(OUTPUT_GROUP_KEY, {}).items()
}

write_json(OUT("insights", "insights_structured.json"), structured)

rejected_df = insights_flat_df[
    ~insights_flat_df["insight_id"].isin(curated_df["insight_id"])
].copy()
write_json(
    OUT("insights", "rejected_insights.json"),
    rejected_df.to_dict(orient="records"),
)

insight_topic_support_df[
    insight_topic_support_df["insight_id"].isin(curated_df["insight_id"])
].to_csv(
    OUT("insights", "insight_topic_support.csv"),
    index=False,
)

chart_cols = [
    "run_id",
    "insight_id",
    "title",
    "section",
    "category_bucket",
    "supporting_project_count",
    "verified_topic_count",
    "mean_topic_share_all_verified_topics",
    "verification_ratio",
    "report_section",
    "report_order",
    "strategic_loop_enabled",
    "strategic_area_id",
    "strategic_area_label",
    "split_id",
    "resolved_groupby_field",
]
chart_ready_insights_df = curated_df[
    [c for c in chart_cols if c in curated_df.columns]
].rename(columns={"title": "theme"})

chart_ready_group_support_df = insight_topic_support_df[
    insight_topic_support_df["insight_id"].isin(curated_df["insight_id"])
].copy()

chart_ready_insights_df.to_csv(
    OUT("chart_data", "chart_ready_insights.csv"),
    index=False,
)
chart_ready_group_support_df.to_csv(
    OUT("chart_data", "chart_ready_group_support.csv"),
    index=False,
)

docx_path = OUT("reports", "trend_tracker_report.docx")
build_packaged_report_docx(
    structured=structured,
    output_path=docx_path,
    output_group_key=OUTPUT_GROUP_KEY,
    report_cfg=CFG["output"],
    project_count=df["project_id"].nunique(),
    run_id=RUN_ID,
)

groups_failed = sorted(set(NMF_GROUPS_FAILED) | set(LABELING_FAILED_GROUPS) | set(SYNTHESIS_FAILED_GROUPS))
eligible_groups = list(per_group_results.keys())
manifest_status = "failure" if groups_failed else "success"

stage_manifest_path = OUT("metadata", "stage_manifest_03_insights_generation.json")
finalize_stage_manifest(
    STAGE_MANIFEST,
    output_path=stage_manifest_path,
    status=manifest_status,
    input_artifacts=[
        artifact_meta(ROOT / "OUTPUTS/prepared/06_enriched.parquet", "enriched_parquet"),
        artifact_meta(ROOT / "OUTPUTS/prepared/metadata/stage_manifest_01_preprocess.json", "stage_manifest_01"),
        artifact_meta(ROOT / "OUTPUTS/enrichment/metadata/stage_manifest_02_semantic_enrichment.json", "stage_manifest_02"),
    ],
    output_artifacts=[
        artifact_meta(COPIED_CONFIG_PATH, "resolved_params_yaml"),
        artifact_meta(FILTER_SPEC_PATH, "filter_spec_json"),
        artifact_meta(FILTER_SUMMARY_PATH, "filter_summary_json"),
        artifact_meta(STRATEGIC_META_PATH, "strategic_run_meta_json"),
        artifact_meta(OUT("analysis", "category_tfidf.csv"), "category_tfidf_csv"),
        artifact_meta(OUT("analysis", "nmf_topics.csv"), "nmf_topics_csv"),
        artifact_meta(OUT("analysis", "nmf_weights.csv"), "nmf_weights_csv"),
        artifact_meta(OUT("analysis", "project_topic_bridge.csv"), "project_topic_bridge_csv"),
        artifact_meta(OUT("analysis", "llm_topic_labels.json"), "llm_topic_labels_json"),
        artifact_meta(OUT("insights", "insights_candidates_draft_pre_metadata.json"), "insights_candidates_draft_pre_metadata_json"),
        artifact_meta(OUT("insights", "metadata_lift_facts_for_step7.csv"), "metadata_lift_facts_for_step7_csv"),
        artifact_meta(OUT("insights", "insights_candidates.json"), "insights_candidates_json"),
        artifact_meta(OUT("insights", "insights_structured.json"), "insights_structured_json"),
        artifact_meta(OUT("insights", "rejected_insights.json"), "rejected_insights_json"),
        artifact_meta(OUT("insights", "insights_flat.csv"), "insights_flat_csv"),
        artifact_meta(OUT("insights", "insight_topic_support.csv"), "insight_topic_support_csv"),
        artifact_meta(OUT("chart_data", "chart_ready_insights.csv"), "chart_ready_insights_csv"),
        artifact_meta(OUT("chart_data", "chart_ready_group_support.csv"), "chart_ready_group_support_csv"),
        artifact_meta(OUT("reports", "trend_tracker_report.docx"), "trend_tracker_report_docx"),
    ],
    row_counts={
        "input_rows": int(len(raw_df)),
        "input_projects": int(raw_df["project_id"].nunique()),
        "filtered_projects_before_strategic_prep": int(BASE_FILTERED_PROJECT_COUNT),
        "run_rows": int(len(df)),
        "run_projects": int(df["project_id"].nunique()),
        "eligible_groups": int(len(eligible_groups)),
        "groups_skipped": int(len(NMF_GROUPS_SKIPPED)),
        "groups_failed": int(len(groups_failed)),
        # v1.4-compatible keys. DEPRECATED after NB04 and audit readers move to v1.5 aliases.
        "topics_generated": int(len(topics_df)),
        "insights_generated": int(len(insights_flat_df)),
        "insights_accepted": int(len(accepted_ids)),
        "insights_rejected": int(len(rejected_ids)),
        # v1.5 aliases
        "topics": int(len(topics_df)),
        "candidate_insights": int(len(insights_flat_df)),
        "accepted_insights": int(len(curated_df)),
        "rejected_insights": int(len(rejected_ids)),
        "metadata_lift_facts_for_step7": int(len(metadata_lift_audit_df)),
    },
    key_params={
        "groupby_field": GROUPBY_FIELD,
        "base_groupby_field": BASE_GROUPBY_FIELD,
        "strategic_loop_enabled": STRATEGIC_LOOP_ENABLED,
        "strategic_run_meta": STRATEGIC_RUN_META,
        "topic_assignment_threshold": CFG["analysis"]["topic_assignment_threshold"],
        "verification_config": VERIFY_CFG,
        "packaging_config": PACKAGING_CFG,
        "dedupe_config": DEDUPE_CFG,
        "metadata_lift_enabled": METADATA_LIFT_ENABLED,
        "metadata_lift_dimensions": METADATA_LIFT_DIMENSIONS,
        "metadata_lift_thresholds": METADATA_LIFT_THRESHOLDS,
        "min_group_projects_effective": MIN_GROUP_PROJECTS_EFFECTIVE,
        "small_slice_mode": SMALL_SLICE_MODE,
        "models": {
            "labeling": MODEL_LABELING,
            "synthesis": MODEL_SYNTHESIS,
            "verify": MODEL_VERIFY,
        },
    },
    warnings_path=WARNINGS_PATH,
)

pipeline_manifest_path = OUT("metadata", "pipeline_manifest.json")
build_pipeline_manifest(
    output_path=pipeline_manifest_path,
    run_id=RUN_ID,
    run_date=RUN_DATE,
    group_by_field=GROUPBY_FIELD,
    filter_spec_path=FILTER_SPEC_PATH,
    filter_summary_path=FILTER_SUMMARY_PATH,
    stage_manifest_paths=[
        ROOT / "OUTPUTS/prepared/metadata/stage_manifest_01_preprocess.json",
        ROOT / "OUTPUTS/enrichment/metadata/stage_manifest_02_semantic_enrichment.json",
        stage_manifest_path,
    ],
    warnings_01_path=ROOT / "OUTPUTS/prepared/metadata/warnings_01.jsonl",
    warnings_02_path=ROOT / "OUTPUTS/enrichment/metadata/warnings_02.jsonl",
    warnings_03_path=WARNINGS_PATH,
    final_outputs={
        # v1.4-compatible keys. DEPRECATED after NB04 and audit readers move to v1.5 aliases.
        "insights_structured_json": str(OUT("insights", "insights_structured.json")),
        "insights_flat_csv": str(OUT("insights", "insights_flat.csv")),
        "insight_topic_support_csv": str(OUT("insights", "insight_topic_support.csv")),
        "chart_ready_insights_csv": str(OUT("chart_data", "chart_ready_insights.csv")),
        "chart_ready_group_support_csv": str(OUT("chart_data", "chart_ready_group_support.csv")),
        "trend_tracker_report_docx": str(docx_path),
        # v1.5 additions / aliases
        "metadata_lift_facts_for_step7_csv": str(OUT("insights", "metadata_lift_facts_for_step7.csv")),
        "insights_structured": str(OUT("insights", "insights_structured.json")),
        "insights_flat": str(OUT("insights", "insights_flat.csv")),
        "insight_topic_support": str(OUT("insights", "insight_topic_support.csv")),
        "metadata_lift_facts_for_step7": str(OUT("insights", "metadata_lift_facts_for_step7.csv")),
        "trend_tracker_report": str(docx_path),
    },
    config_path=MANIFEST_CFG_PATH,
    filter_fields_key=FILTER_FIELDS_KEY,
    status=manifest_status,
)

print(f"Final structured insights: {OUT('insights', 'insights_structured.json')}")
print(f"DOCX report: {docx_path}")
print(f"Stage manifest: {stage_manifest_path}")
print(f"Pipeline manifest: {pipeline_manifest_path}")
print(f"Manifest status: {manifest_status}")
if groups_failed:
    print(f"Groups with partial failures: {groups_failed}")
